## PaperUno_Figure_SI

In [1]:
%matplotlib inline
import re
import sys
from pathlib import Path

import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import gridspec, patches, ticker
from matplotlib.colorbar import Colorbar
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize, PowerNorm, to_rgba


In [2]:
# Optimized Import and Setup
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Add Figure directory to path (relative to this notebook)
# Assuming notebook is in Figure/PaperUno/ and helper scripts are in Figure/
sys.path.append(str(Path.cwd().parent))

from config import setup, DATA_ROOT, OUTPUT_ROOT

# Configure plotting style and colors
colors = setup()

# Set output directory
path_out = OUTPUT_ROOT

### Figure R1-31, HAADF of EMD of SK

In [3]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 10},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [4]:
# 读取数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\ExSitu\αMnO2\XAS\EMD-SK\Echem\A6-FC\A6"  # noqa: E501, RUF001
)

file1 = plt.imread(Path.joinpath(path_file, r"A6acFC-97000X-0001.png"))
file2 = plt.imread(Path.joinpath(path_file, r"A6acFC-400000X-0004.png"))
file3 = hs.load(Path.joinpath(path_file, r"A6acFC-300X-0009.dm3")) # pyright: ignore[reportCallIssue]
file4 = pd.read_csv(Path.joinpath(path_file, r"profile.txt"), sep="\t", header=0,)

In [ ]:
file4['Distance (nm)'] = 1/file4['g [1/nm]']

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)

img = file1[250:, 250:, :]
ax.imshow(img, cmap="gray")
add_sizebar(ax, 20, 0.11194179207086563, "white", "nm")
ax.set_axis_off()

ax.text(
    -0.03, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_axes((0.1, 0, 1, 1), zorder=0)

img = file2[:1800, :1800, :]
ax.imshow(img, cmap="gray")
add_sizebar(ax, 5, 0.026372302323579788, "white", "nm")
ax.set_axis_off()

ax.text(
    -0.03, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_axes((0.2, 0, 1, 1), zorder=1)

img = file3.data.copy()
ax.imshow(img, cmap="gray", norm=PowerNorm(gamma=0.4, vmin=img.min(), vmax=img.max()))
add_sizebar(ax, 5, file3, "white")
ax.set_axis_off()

ax.text(
    0.75, 0.42, "0.140 nm", transform=ax.transAxes, fontsize=8, va="bottom", ha="center", fontfamily="Arial", fontweight="bold", color='w'
)
ax.text(
    0.6, 0.49, "0.238 nm", transform=ax.transAxes, fontsize=8, va="bottom", ha="center", fontfamily="Arial", fontweight="bold", color='y'
)

def add_arc(ax, center, r_px, theta1=0, theta2=90,  # noqa: PLR0913
            color="cyan", lw=2.0, ls="solid", alpha=1.0):

    arc = patches.Arc(center, alpha=alpha,
                      width=2*r_px, height=2*r_px,  # 宽高=直径
                      theta1=theta1, theta2=theta2,
                      edgecolor=color, linewidth=lw, linestyle=ls)
    ax.add_patch(arc)
    return arc

# 示例：三条不同样式的环
cx, cy = img.shape[1] / 2 - 30, img.shape[0] / 2 - 10
# add_arc(ax, (cx, cy), r_px=1, theta1=0, theta2=360, color="yellow", lw=2.5, ls="--")
add_arc(ax, (cx, cy), r_px=290, theta1=240, theta2=340, color="yellow", lw=1.0, ls="--", alpha=0.5)
add_arc(ax, (cx, cy), r_px=500, theta1=240, theta2=5, color="w", lw=1.0, ls="--", alpha=0.5)

ax.text(
    -0.03, 1.0, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_31_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_31_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_31_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_31_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure R1-11, Operando pH

In [ ]:
path_filelist = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH2_02C_CP_MnO2_pH_0_664mg").glob( # noqa: E501, RUF001
        r"*GCPL*.txt"
    )
)
# 读取电化学数据
echem = []  # 用于存储所有的电化学数据
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i].iloc[:, 0].isin([0])]

In [ ]:
path_file = Path(DATA_ROOT / r'Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH2_02C_CP_MnO2_pH_0_664mg')  # noqa: E501
data = pd.read_csv(
    path_file.joinpath(r"pH measurement.csv"),
    sep=r',',
    comment="#",
    skiprows=0,
    usecols=[0, 3],
    encoding="latin_1",
    engine="python",
    index_col=None,
    header=0,
    decimal=".",
    names=["time/s", "pH"]
)
# 清洗数据
data['pH'] = pd.to_numeric(data['pH'], errors='coerce')
data['time/s'] = pd.to_datetime(data['time/s'])

In [ ]:
echem[0] = echem[0][echem[0]['Ewe/V'] > 0.55]  # 只保留正电压数据

In [ ]:
tolerance = pd.Timedelta(seconds=1)
data_matched = pd.merge_asof(echem[0], data, on='time/s', direction='nearest', tolerance=tolerance)
idx_min = data_matched["Ewe/V"].idxmin()
# 截取：只保留最小值之前的行
data_matched = data_matched.loc[:idx_min].copy()
data_matched

In [ ]:
path_filelist = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH5_03C_CP_MnO2_pH_0mg304").glob( # noqa: E501, RUF001
        r"*GCPL*.txt"
    )
)
# 读取电化学数据
echem1 = []  # 用于存储所有的电化学数据
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem1.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem1)):
    echem1[i] = echem1[i][echem1[i].iloc[:, 0].isin([0])]

In [ ]:
path_file = Path(DATA_ROOT / r'Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH5_03C_CP_MnO2_pH_0mg304')  # noqa: E501
data1 = pd.read_csv(
    path_file.joinpath(r"pH5Ta.csv"),
    sep=r',',
    comment="#",
    skiprows=0,
    usecols=[0, 3],
    encoding="latin_1",
    engine="python",
    index_col=None,
    header=0,
    decimal=".",
    names=["time/s", "pH"]
)
# 清洗数据
data1['pH'] = pd.to_numeric(data1['pH'], errors='coerce')
data1['time/s'] = pd.to_datetime(data1['time/s'])

In [ ]:
echem1[0] = echem1[0][echem1[0]['Ewe/V'] > 0.15]  # 只保留正电压数据

In [ ]:
tolerance = pd.Timedelta(seconds=1)
data_matched1 = pd.merge_asof(echem1[0], data1, on='time/s', direction='nearest', tolerance=tolerance)
idx_min = data_matched1["Ewe/V"].idxmin()
# 截取：只保留最小值之前的行
data_matched1 = data_matched1.loc[:idx_min].copy()
data_matched1

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1,1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)
labels = [[r"$\mathrm{1 \ M \ ZnSO_4 \ + \ 0.2 \ M \ MnSO_4, 0.304 \ mg}$", None],[None, None]]  # noqa: E501
mass_loadings1 = [0.304/1000,]
for i, data in enumerate([data_matched1,]):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "Capacity/mA.h"]/mass_loadings1[i],
            temp.loc[: idx_min - 1, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[i],
            label=labels[i][j],
            zorder=0,
        )

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-5, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))

ax.set_ylabel(r"WE, Potential (V vs. Ag/AgCl)", fontsize=9)
ax.set_ylim(0.1, 0.7)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

ax.tick_params(axis="both", direction="out", labelsize=9)

ax.text(
    0.05,
    0.14,
    r"$\mathrm{C/3, 1 \ C = 308 \ mA \ g^{-1}}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c='k',
)
# ax.text(
#     0.05,
#     0.14,
#     r"$\mathrm{pH@5.0}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c='k',
# )
arrowprops = {
    "arrowstyle": "<-",
    "color": colors[0],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "-",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.2, 0.35), xycoords="axes fraction", xytext=(0, 0.25), textcoords="axes fraction", arrowprops=arrowprops
)

# 右边
ax2 = ax.twinx()
ax2.set_position((0, 0, 1.0, 1.0))
ax2.set_box_aspect(0.8)

ax2.plot(data_matched1['Capacity/mA.h']/mass_loadings1[0], data_matched1['pH'], ls="--", lw=1.0, c=colors[1], label='Operando pH@5.0', zorder=0)  # noqa: E501

ax2.set_ylabel(r"pH Values", fontsize=9)
ax2.set_ylim(4.0, 5.2)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

arrowprops = {
    "arrowstyle": "<-",
    "color": colors[1],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "--",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.8, 0.75), xycoords="axes fraction", xytext=(1.0, 0.65), textcoords="axes fraction", arrowprops=arrowprops
)

# 图例
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax.legend(handles1 + handles2, labels1 + labels2,
          loc="upper right", bbox_to_anchor=(0.8, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)

ax.text(
    -0.18, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.25, 0, 1, 1))
ax.set_box_aspect(0.8)
labels = [[r"$\mathrm{1 \ M \ ZnSO_4 \ + \ 0.2 \ M \ MnSO_4, 0.664 \ mg}$", None],[None, None]]  # noqa: E501
mass_loadings = [0.664/1000,]
for i, data in enumerate([data_matched,]):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]

        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "Capacity/mA.h"]/mass_loadings[i],
            temp.loc[: idx_min - 1, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[i],
            label=labels[i][j],
            zorder=0,
        )

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-5, 240)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=40, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax.set_ylabel(r"WE, Potential (V vs. Ag/AgCl)", fontsize=9)
ax.set_ylim(0.5, 1.1)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

ax.tick_params(axis="both", direction="out", labelsize=9)

ax.text(
    0.05,
    0.14,
    r"$\mathrm{C/5, 1 \ C = 308 \ mA \ g^{-1}}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c='k',
)
# ax.text(
#     0.05,
#     0.14,
#     r"$\mathrm{pH@2.0}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c='k',
# )
arrowprops = {
    "arrowstyle": "<-",
    "color": colors[0],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "-",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.2, 0.45), xycoords="axes fraction", xytext=(0, 0.35), textcoords="axes fraction", arrowprops=arrowprops
)

# 右边
ax2 = ax.twinx()
ax2.set_position((0.25, 0, 1.0, 1.0))
ax2.set_box_aspect(0.8)

ax2.plot(data_matched['Capacity/mA.h']/mass_loadings[0], data_matched['pH'], ls="--", lw=1.0, c=colors[1], label='Operando pH@2.0', zorder=0)  # noqa: E501

ax2.set_ylabel(r"pH Values", fontsize=9)
ax2.set_ylim(1.0, 2.2)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

arrowprops = {
    "arrowstyle": "<-",
    "color": colors[1],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "--",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.8, 0.75), xycoords="axes fraction", xytext=(1.0, 0.65), textcoords="axes fraction", arrowprops=arrowprops
)

# 图例
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax.legend(handles1 + handles2, labels1 + labels2,
          loc="upper right", bbox_to_anchor=(0.8, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)

ax.text(
    -0.18, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.gcf().set_facecolor('white')
plt.show()

In [ ]:
path_filelist = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH2_02C_CP_MnO2_pH_0_664mg").glob( # noqa: E501, RUF001
        r"*GCPL*.txt"
    )
)
# 读取电化学数据
echem = []  # 用于存储所有的电化学数据
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i].iloc[:, 0].isin([0])]

In [ ]:
path_file = Path(DATA_ROOT / r'Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH2_02C_CP_MnO2_pH_0_664mg')  # noqa: E501
data = pd.read_csv(
    path_file.joinpath(r"pH measurement.csv"),
    sep=r',',
    comment="#",
    skiprows=0,
    usecols=[0, 3],
    encoding="latin_1",
    engine="python",
    index_col=None,
    header=0,
    decimal=".",
    names=["time/s", "pH"]
)
# 清洗数据
data['pH'] = pd.to_numeric(data['pH'], errors='coerce')
data['time/s'] = pd.to_datetime(data['time/s'])

In [ ]:
echem[0] = echem[0][echem[0]['Ewe/V'] > 0.55]  # 只保留正电压数据

In [ ]:
tolerance = pd.Timedelta(seconds=1)
data_matched = pd.merge_asof(echem[0], data, on='time/s', direction='nearest', tolerance=tolerance)
idx_min = data_matched["Ewe/V"].idxmin()
# 截取：只保留最小值之前的行
data_matched = data_matched.loc[:idx_min].copy()
data_matched

In [ ]:
path_filelist = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH5_03C_CP_MnO2_pH_0mg304").glob( # noqa: E501, RUF001
        r"*GCPL*.txt"
    )
)
# 读取电化学数据
echem1 = []  # 用于存储所有的电化学数据
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem1.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem1)):
    echem1[i] = echem1[i][echem1[i].iloc[:, 0].isin([0])]

In [ ]:
path_file = Path(DATA_ROOT / r'Zn-Mn\PaperUno\pH\Operando\1M ZnSO4 + 0.2 M MnSO4\1M+02M+pH5_03C_CP_MnO2_pH_0mg304')  # noqa: E501
data1 = pd.read_csv(
    path_file.joinpath(r"pH5Ta.csv"),
    sep=r',',
    comment="#",
    skiprows=0,
    usecols=[0, 3],
    encoding="latin_1",
    engine="python",
    index_col=None,
    header=0,
    decimal=".",
    names=["time/s", "pH"]
)
# 清洗数据
data1['pH'] = pd.to_numeric(data1['pH'], errors='coerce')
data1['time/s'] = pd.to_datetime(data1['time/s'])

In [ ]:
echem1[0] = echem1[0][echem1[0]['Ewe/V'] > 0.15]  # 只保留正电压数据

In [ ]:
tolerance = pd.Timedelta(seconds=1)
data_matched1 = pd.merge_asof(echem1[0], data1, on='time/s', direction='nearest', tolerance=tolerance)
idx_min = data_matched1["Ewe/V"].idxmin()
# 截取：只保留最小值之前的行
data_matched1 = data_matched1.loc[:idx_min].copy()
data_matched1

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1,1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)
labels = [[r"$\mathrm{1 \ M \ ZnSO_4 \ + \ 0.2 \ M \ MnSO_4, 0.304 \ mg}$", None],[None, None]]  # noqa: E501
mass_loadings1 = [0.304/1000,]
for i, data in enumerate([data_matched1,]):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "Capacity/mA.h"]/mass_loadings1[i],
            temp.loc[: idx_min - 1, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[i],
            label=labels[i][j],
            zorder=0,
        )

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-5, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))

ax.set_ylabel(r"WE, Potential (V vs. Ag/AgCl)", fontsize=9)
ax.set_ylim(0.1, 0.7)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

ax.tick_params(axis="both", direction="out", labelsize=9)

ax.text(
    0.05,
    0.14,
    r"$\mathrm{C/3, 1 \ C = 308 \ mA \ g^{-1}}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c='k',
)
# ax.text(
#     0.05,
#     0.14,
#     r"$\mathrm{pH@5.0}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c='k',
# )
arrowprops = {
    "arrowstyle": "<-",
    "color": colors[0],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "-",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.2, 0.35), xycoords="axes fraction", xytext=(0, 0.25), textcoords="axes fraction", arrowprops=arrowprops
)

# 右边
ax2 = ax.twinx()
ax2.set_position((0, 0, 1.0, 1.0))
ax2.set_box_aspect(0.8)

ax2.plot(data_matched1['Capacity/mA.h']/mass_loadings1[0], data_matched1['pH'], ls="--", lw=1.0, c=colors[1], label='Operando pH@5.0', zorder=0)  # noqa: E501

ax2.set_ylabel(r"pH Values", fontsize=9)
ax2.set_ylim(4.0, 5.2)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

arrowprops = {
    "arrowstyle": "<-",
    "color": colors[1],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "--",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.8, 0.75), xycoords="axes fraction", xytext=(1.0, 0.65), textcoords="axes fraction", arrowprops=arrowprops
)

# 图例
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax.legend(handles1 + handles2, labels1 + labels2,
          loc="upper right", bbox_to_anchor=(0.8, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)

ax.text(
    -0.18, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.25, 0, 1, 1))
ax.set_box_aspect(0.8)
labels = [[r"$\mathrm{1 \ M \ ZnSO_4 \ + \ 0.2 \ M \ MnSO_4, 0.664 \ mg}$", None],[None, None]]  # noqa: E501
mass_loadings = [0.664/1000,]
for i, data in enumerate([data_matched,]):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]

        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "Capacity/mA.h"]/mass_loadings[i],
            temp.loc[: idx_min - 1, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[i],
            label=labels[i][j],
            zorder=0,
        )

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-5, 240)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=40, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))

ax.set_ylabel(r"WE, Potential (V vs. Ag/AgCl)", fontsize=9)
ax.set_ylim(0.5, 1.1)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

ax.tick_params(axis="both", direction="out", labelsize=9)

ax.text(
    0.05,
    0.14,
    r"$\mathrm{C/5, 1 \ C = 308 \ mA \ g^{-1}}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c='k',
)
# ax.text(
#     0.05,
#     0.14,
#     r"$\mathrm{pH@2.0}$",
#     ha="left",
#     va="top",
#     transform=ax.transAxes,
#     fontsize=9,
#     c='k',
# )
arrowprops = {
    "arrowstyle": "<-",
    "color": colors[0],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "-",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.2, 0.45), xycoords="axes fraction", xytext=(0, 0.35), textcoords="axes fraction", arrowprops=arrowprops
)

# 右边
ax2 = ax.twinx()
ax2.set_position((0.25, 0, 1.0, 1.0))
ax2.set_box_aspect(0.8)

ax2.plot(data_matched['Capacity/mA.h']/mass_loadings[0], data_matched['pH'], ls="--", lw=1.0, c=colors[1], label='Operando pH@2.0', zorder=0)  # noqa: E501

ax2.set_ylabel(r"pH Values", fontsize=9)
ax2.set_ylim(1.0, 2.2)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))

arrowprops = {
    "arrowstyle": "<-",
    "color": colors[1],
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "--",
    "alpha": 0.8,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.8, 0.75), xycoords="axes fraction", xytext=(1.0, 0.65), textcoords="axes fraction", arrowprops=arrowprops
)

# 图例
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax.legend(handles1 + handles2, labels1 + labels2,
          loc="upper right", bbox_to_anchor=(0.8, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)

ax.text(
    -0.18, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.gcf().set_facecolor('white')
plt.show()

### Figure R1-05, Echem in 1 M + 1 M, 0.5M + 0.2 M, 2M + 0.2 M

In [ ]:
# 电化学上的时间
path_filelist = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\GCD\1M ZnSO4 + 1M MnSO4").glob(
        r"LC*.xlsx"
    )
)
echem = []
# 读取电化学数据
for file in path_filelist:
    df = pd.read_excel(
        file,
        sheet_name=2,
        engine="openpyxl",
        comment="#",
        header=0,
        index_col=None,
    ).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

path_filelist = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\PassivatedZnRichCoatting\DifferentElectrolytes\0.5M ZnSO4 + 0.2M MnSO4\05M+02M_02C_MnO2_5_2mg").glob( # noqa: E501, RUF001
        r"**\*.txt"
    )
)
# 读取电化学数据
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

path_filelist = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\PassivatedZnRichCoatting\DifferentElectrolytes\2M ZnSO4 + 0.2M MnSO4\2M+02M_02C_MnO2_5_1mg").glob( # noqa: E501, RUF001
        r"**\*.txt"
    )
)
# 读取电化学数据
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    if i == 0:
        echem[i] = echem[i][echem[i].iloc[:, 0].isin([0, 1])]
    else:
        echem[i] = echem[i][echem[i].iloc[:, 0].isin([0])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)
labels = [[r"$\mathrm{1 \ M \ ZnSO_4 \ + \ 1 \ M \ MnSO_4, 1.96 \ mg \ cm^{-2}}$", None], [r"$\mathrm{0.5 \ M \ ZnSO_4 \ + \ 0.2 \ M \ MnSO_4, 0.84 \ mg \ cm^{-2}}$", None], [r"$\mathrm{2 \ M \ ZnSO_4 \ + \ 0.2 \ M \ MnSO_4, 0.68 \ mg \ cm^{-2}}$", None]]  # noqa: E501
mass_loadings = [1.0, 0.424/1000, 0.343/1000]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        if i == 0:
            idx_min = temp["Voltage/V"].idxmin()
            # 断开最小值前后的曲线
            ax.plot(
                temp.loc[: idx_min - 1, "SpeCap/mAh/g"]/mass_loadings[i],
                temp.loc[: idx_min - 1, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[i],
                label=labels[i][j],
                zorder=0,
            )
        else:
            idx_min = temp["Ewe/V"].idxmin()
            # 断开最小值前后的曲线
            ax.plot(
                temp.loc[: idx_min - 1, "Capacity/mA.h"]/mass_loadings[i],
                temp.loc[: idx_min - 1, "Ewe/V"],
                ls="-",
                lw=1.0,
                c=colors[i],
                label=labels[i][j],
                zorder=0,
            )


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(-0.1, 350)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.85, 1.65)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.text(
    0.05,
    0.20,
    r"C/10",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[0],
)
ax.text(
    0.15,
    0.20,
    r"C/5",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[1],
)
ax.text(
    0.23,
    0.20,
    r"C/5",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[2],
)
ax.text(
    0.05,
    0.14,
    r"$\mathrm{1 \ C = 308 \ mA \ g^{-1}}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c='k',
)
# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.gcf().set_facecolor('white')
plt.show()

### Figure SI_27, Echem of EMD_SK

In [ ]:
# Saivash Data
path_saivash = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\ExSitu\αMnO2\XAS\EMD-SK\Echem\HaC FC\HAc FD_C01.txt"
)  # noqa: E501, RUF001

# 读取电化学数据
with open(path_saivash, "r", encoding="latin_1") as file:
    for line in file:
        if line.startswith("Nb header lines"):
            line_skip = int(line.split(":")[1].strip())
            break  # 发现后立即退出循环，提高效率  # noqa: RUF003

df = pd.read_csv(
    path_saivash,
    sep=r"\t",
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    index_col=None,
    decimal=".",
    engine="python",
).dropna(axis=1, how="all")
# # 转换数据格式
df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(
    pd.to_numeric, errors="coerce"
)
df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
data = df.iloc[:, [0, 1, 2, 4, 5]]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [r"$\mathrm{Electrolytic \ MnO_2}$", None, None, None]

for i, idx in enumerate(data["cycle number"].unique()):
    temp = data[data.iloc[:, 0] == idx]
    # 找到电压最小值的索引
    idx_min = temp[temp["<I>/mA"] < 0]["Ewe/V"].idxmax()
    # 断开最小值前后的曲线
    ax.plot(
        temp.loc[: idx_min - 1, r"Capacity/mA.h"] / 9,
        temp.loc[: idx_min - 1, r"Ewe/V"],
        ls="-",
        lw=1.0,
        c=colors[i],
        label=labels[i],
        zorder=0,
    )  # type: ignore  # noqa: E501
    ax.plot(
        temp.loc[idx_min + 1 : 444, r"Capacity/mA.h"] / 9,
        temp.loc[idx_min + 1 : 444, r"Ewe/V"],
        ls="-",
        lw=1.0,
        c=colors[i],
        label=None,
        zorder=0,
    )  # type: ignore # noqa: E501

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,cm^{-2}_{MnO_2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(0.0, 2.4)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.5, 2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(0.03, 0.20, r"$\mathrm{CE: \ 91.6\%}$", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")  # noqa: E501
ax.text(
    0.03,
    0.15,
    r"$\mathrm{1\ M \ pH \ Buffer, CH_3COOH+NH_3, \ pH=4.20}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c="k",
)  # noqa: E501
ax.text(
    0.03,
    0.1,
    r"$\mathrm{Folw Cell, \ 0.5\ M \ ZnSO_4 + 0.2\ M \ MnSO_4}, 10 \ mA \ cm^{-2}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c="k",
)  # noqa: E501

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_27_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_27_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_27_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_27_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 26 PCA for STXM

In [ ]:
# 数据导入
path_file = Path(DATA_ROOT / r"Zn-Mn\PaperUno\STXM\ExSitu\Andrea\SpectrumAll")
pca = pd.read_excel(
    Path.joinpath(path_file, "sTXM_MCR-ALS-3comp.xlsx"),
    sheet_name=1,
    index_col=None,
    header=0,
    comment="#",
    engine="openpyxl",
)

In [ ]:
# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0.0, hspace=0.0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(pca.iloc[:, 0], pca.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label=r"#1")  # type: ignore
ax.plot(pca.iloc[:, 0], pca.iloc[:, 2], ls="-", lw=1.0, c=colors[1], label=r"#2")  # type: ignore
ax.plot(pca.iloc[:, 0], pca.iloc[:, 3], ls="-", lw=1.0, c=colors[2], label=r"#3")  # type: ignore
ax.legend(loc="upper right", bbox_to_anchor=(0.98, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

ax.set_xlim(630, 670)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0.0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0.0))
ax.set_ylim(-0.1, 3.0)
ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0.0))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_26_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_26_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_26_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_26_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 25 TEM-EELS of discharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, data, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / data.axes_manager["y"].scale,
        "{} {}".format(size, data.axes_manager["y"].units),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
        fontproperties={"size": 10.0},
    )
    ax.add_artist(asb)


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 读取数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS"  # noqa: E501, RUF001
)
haddf = hs.load(Path.joinpath(path_file, r"Data", r"SI2_80pA_10ms", r"STEM SI.dm4"))  # type: ignore
labels = hs.load(
    Path.joinpath(path_file, r"Results", r"SI2_80pA_10ms", r"Hyperspy", r"V1", r"6-CS_cluster_labels_Mn_2.hspy")
)  # type: ignore  # noqa: E501
signal_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Results", r"SI2_80pA_10ms", r"Hyperspy", r"V1", r"1st_09V_Mn_Clusters.nor"),
    sep=r"\s+",
    comment="#",
    header=0,
    index_col=None,
)
signal_O = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Results", r"SI2_80pA_10ms", r"Hyperspy", r"V1", r"1st_09V_O_Clusters.nor"),
    sep=r"\s+",
    comment="#",
    header=0,
    index_col=None,
)
onsetenergy = hs.load(
    Path.joinpath(path_file, r"Results", r"SI2_80pA_10ms", r"Hyperspy", r"V1", r"5-ps_onset_energy_Mn_L3_b.hspy")
)  # type: ignore  # noqa: E501

path_file_pris = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2024-EMCA\EELS"  # noqa: RUF001
)
signal_Mn_pris = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file_pris, r"Results", r"SI1_150pA_10ms", r"Hyperspy", r"V1", r"Pristine_Mn_Clusters.nor"),
    sep=r"\s+",
    comment="#",
    header=0,
    index_col=None,
)
signal_O_pris = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file_pris, r"Results", r"SI1_150pA_10ms", r"Hyperspy", r"V1", r"Pristine_O_Clusters.nor"),
    sep=r"\s+",
    comment="#",
    header=0,
    index_col=None,
)

In [ ]:
haddf[1] = haddf[1].isig[10240:, :]
labels = labels.isig[160:, :]
onsetenergy = onsetenergy.inav[160:, :]

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1.0), zorder=0)
ax.set_aspect(1.0)

ax.imshow(haddf[1].data, cmap="gray")
add_sizebar(ax, 20, haddf[1], "w")
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=10,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_axes((-0.3, 0, 1, 1), zorder=0)
ax.set_aspect(1.0)

ax.imshow(labels.inav[0].data, cmap="gray")
add_sizebar(ax, 20, labels.inav[0], "w")
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"Bulk Cluster",
    transform=ax.transAxes,
    fontsize=10,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_axes((-0.6, 0, 1, 1), zorder=0)
ax.set_aspect(1.0)

ax.imshow(labels.inav[1].data, cmap="gray")
add_sizebar(ax, 20, labels.inav[1], "w")
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"Surface Cluster",
    transform=ax.transAxes,
    fontsize=10,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1, 1.0, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 E
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_axes((0.45, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)
Clusters = ["Bulk", "Surface"]
for i in range(signal_O.shape[1] - 1):
    ax.plot(signal_O.iloc[:, 0], signal_O.iloc[:, i + 1], c=colors[i], ls="-", lw=1.0, label=Clusters[i], zorder=i)  # type: ignore
    ax.plot(
        signal_O_pris.iloc[:, 0],
        signal_O_pris.iloc[:, i + 1],
        c=colors[i],  # type: ignore
        ls="--",
        lw=1.0,
        label=None,
        zorder=i,
        alpha=0.8,
    )
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.50, 0.30),
    ncols=1,
    frameon=False,
    fontsize=11,
    labelcolor="linecolor",
    columnspacing=0.4,
)
ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=9)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(510.0, 590.0)
ax.set_ylim(-0.03, 1.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))

ax.text(0.03, 0.98, r"O $\mathit{K}$ edge", transform=ax.transAxes, fontsize=11, va="top", fontfamily="Arial")
ax.text(
    -0.25, 1.0, "e", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 F
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_axes((0.85, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

for i in range(signal_Mn.shape[1] - 1):
    ax.plot(signal_Mn.iloc[:, 0], signal_Mn.iloc[:, i + 1], c=colors[i], ls="-", lw=1.0, label=Clusters[i], zorder=i)  # type: ignore
    ax.plot(
        signal_Mn_pris.iloc[:, 0],
        signal_Mn_pris.iloc[:, i + 1],
        c=colors[i],  # type: ignore
        ls="--",
        lw=1.0,
        label=None,
        zorder=i,
        alpha=0.8,
    )

ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=9)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(620.0, 700.0)
ax.set_ylim(-0.03, 3.6)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.9, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.45, offset=0))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.50, 0.30),
    ncols=1,
    frameon=False,
    fontsize=11,
    labelcolor="linecolor",
    columnspacing=0.4,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(0.03, 0.98, r"Mn $\mathit{L}$ edge", transform=ax.transAxes, fontsize=11, va="top", fontfamily="Arial")
ax.text(
    -0.22, 1.0, "f", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.50, 0.55, 0.45, 0.4), ylim=(1.8, 3.0), xlim=(637.0, 643.0))

for i in range(signal_Mn.shape[1] - 1):
    axins.plot(signal_Mn.iloc[:, 0], signal_Mn.iloc[:, i + 1], c=colors[i], ls="-", lw=1.0, label=Clusters[i], zorder=i)  # type: ignore
    axins.plot(
        signal_Mn_pris.iloc[:, 0],
        signal_Mn_pris.iloc[:, i + 1],
        c=colors[i],  # type: ignore
        ls="--",
        lw=1.0,
        label=None,
        zorder=i,
        alpha=0.8,
    )

axins.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=-1))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-1))

axins.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=False,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=False,
    labelright=False,
)
axins.text(
    0.9,
    0.8,
    r"$\mathrm{L_3}$",
    transform=axins.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_axes((0.1, 0, 1, 1), zorder=0)
# ax.set_box_aspect(0.8)  # noqa: ERA001
vmin_L3 = 635.1  # onsetenergy.nanmin().data[0]  # noqa: N816
vmax_L3 = 635.5  # onsetenergy.nanmax().data[0]  # noqa: N816
N_color = 5
# 创建 colormap 并设置 over/under 颜色
cmap = plt.cm.hot_r(np.linspace(0.2, 0.8, 256))  # type: ignore
cmap = LinearSegmentedColormap.from_list("cmap", cmap)
cmap.set_over("white")  # 超出 vmax 显示白色
cmap.set_under("black")  # 低于 vmin 显示黑色

onsetenergy_a = np.nan_to_num(onsetenergy.data, copy=True, nan=0)
# 设置 norm
norm = mpl.colors.Normalize(vmin=vmin_L3, vmax=vmax_L3)  # type: ignore

ima = ax.imshow(onsetenergy_a, cmap=cmap, norm=norm)
add_sizebar(ax, 20, onsetenergy, "w")
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
cax = subfig.add_subplot()
cax.set_position((0.9, 0, 0.08, 0.8))
subfig.colorbar(
    mappable=ima,
    cax=cax,
    ticks=np.linspace(vmin_L3, vmax_L3, N_color),
    format="{x:.1f}",
    location="right",
    orientation="vertical",
)

cax.tick_params(axis="x", direction="out")
cax.text(
    -0.02,
    1.2,
    r"Energy (eV)",
    horizontalalignment="left",
    verticalalignment="top",
    transform=cax.transAxes,
    fontsize=10,
    c="k",
    rotation="horizontal",
)
ax.text(
    -0.05, 1.0, "d", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_25_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_25_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_25_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_25_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### FigureSI_24 XPS of the discharged sample

In [ ]:
# 读取数据
path_xps = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XPS\ExSitu\αMnO2\Charge\Electrode\1st0.9V\1M ZnSO4 + 0.2M MnSO4\BulkCell\Results"  # noqa: E501
)
Mn_xps = pd.read_fwf(
    Path.joinpath(path_xps, r"2025_085_sample7_Mn.txt"),
    sep=r"\s+",
    skiprows=8,
    index_col=None,
    header=None,
    comment="#",
)
Mn_xps.dropna(axis=1, how="all", inplace=True)

In [ ]:
# 数据处理
envelope_Mn_IV = Mn_xps.iloc[:, 30:38].sum(axis=1) - Mn_xps.iloc[:, 54] * 7  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 38:46].sum(axis=1) - Mn_xps.iloc[:, 54] * 7  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 46:54].sum(axis=1) - Mn_xps.iloc[:, 54] * 7  # noqa: N816

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 28], Mn_xps.iloc[:, 29], label=r"Discharged", color=colors[0])  # type: ignore # each peak of Mn_xps_data
ax.plot(
    Mn_xps.iloc[:, 28],
    Mn_xps.iloc[:, 55],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)  # envelope_all
ax.plot(Mn_xps.iloc[:, 28], Mn_xps.iloc[:, 54], label=r"Background", color=colors[2], zorder=3)  # type: ignore # background

ax.plot(
    Mn_xps.iloc[:, 28],
    envelope_Mn_IV,
    color=colors[3],
    linestyle="--",
    label=r"$\mathrm{Mn^{IV}}$",  # type: ignore
)  # envelope_Mn_IV
ax.plot(
    Mn_xps.iloc[:, 28],
    envelope_Mn_III,
    color=colors[5],
    linestyle="--",
    label=r"$\mathrm{Mn^{III}}$",  # type: ignore
)  # envelope_Mn_III
ax.plot(
    Mn_xps.iloc[:, 28],
    envelope_Mn_II,
    color=colors[4],
    linestyle="--",
    label=r"$\mathrm{Mn^{II}}$",
    zorder=3,  # type: ignore
)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 28], Mn_xps.iloc[:, 30:38], color=colors[3], alpha=0.5)  # type: ignore # each peak of Mn_xps_Mn_IV
for i in range(8):
    ax.fill_between(
        x=Mn_xps.iloc[:, 28],
        y1=Mn_xps.iloc[:, i + 30],
        y2=Mn_xps.iloc[:, 54],
        facecolor=colors[3],
        alpha=0.3,  # type: ignore
    )
ax.plot(Mn_xps.iloc[:, 28], Mn_xps.iloc[:, 38:46], color=colors[5], alpha=0.5)  # type: ignore # each peak of Mn_xps_Mn_III
for i in range(8):
    ax.fill_between(
        x=Mn_xps.iloc[:, 28],
        y1=Mn_xps.iloc[:, i + 38],
        y2=Mn_xps.iloc[:, 54],
        facecolor=colors[5],
        alpha=0.3,  # type: ignore
    )
ax.plot(Mn_xps.iloc[:, 28], Mn_xps.iloc[:, 46:54], color=colors[4], alpha=0.5)  # type: ignore # each peak of Mn_xps_Mn_II
for i in range(8):
    ax.fill_between(
        x=Mn_xps.iloc[:, 28],
        y1=Mn_xps.iloc[:, i + 46],
        y2=Mn_xps.iloc[:, 54],
        facecolor=colors[4],
        alpha=0.3,  # type: ignore
    )

ax.set_xlabel(r"Binding Energy (eV)", fontsize=9)
ax.set_xlim(649.0, 639.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(1500, 5500)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=800, offset=-100))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=400, offset=-100))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(axis="both", labelsize=9)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.02),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)
ax.text(
    0.03,
    0.70,
    r"$\mathrm{Mn^{III}}$:",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[5],  # type: ignore
)
ax.text(
    0.03,
    0.62,
    r"61.54 at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[5],  # type: ignore
)
ax.text(
    0.03,
    0.50,
    r"$\mathrm{Mn^{IV}}$:",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[3],  # type: ignore
)
ax.text(
    0.03,
    0.42,
    r"38.46 at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[3],  # type: ignore
)
ax.text(
    0.78,
    0.87,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=11,
)
# ax.text(-0.21, 1.0, 'a', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_24_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_24_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_24_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_24_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Fugire_SI 23 sTXM in Poland

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, scale, unit, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / scale,
        "{} {}".format(size, unit),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
    )
    scalebar = ax.add_artist(asb)
    return scalebar

In [ ]:
file_folder = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\STXM\InSitu\αMnO2\March_2024_CHIP9\Results\Echem\Charge"
)
image_path = Path(DATA_ROOT / r"Zn-Mn\PaperUno\STXM\InSitu\αMnO2\March_2024_CHIP9\Results\OP")
sTXM_path = Path(  # noqa: N816
    DATA_ROOT / r"Zn-Mn\PaperUno\STXM\InSitu\αMnO2\March_2024_CHIP9\Results\EnergyScan"
)
image_path2 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\STXM\InSitu\αMnO2\March_2024_CHIP9\Results\Images"
)
stxm_mn = pd.read_csv(Path.joinpath(sTXM_path, r"Mn_Charge.nor"), sep=r"\s+", index_col=False, header=0, comment="#")
image_list = list(image_path2.glob(r"*.tif"))

In [ ]:
file_ocv_list = list(file_folder.glob(r"*OCV.txt"))
file_gcpl_list = list(file_folder.glob(r"*GCPL.txt"))
file_ca_list = list(file_folder.glob(r"*C*A.txt"))

df_ocv = []
df_gcpl = []
df_ca = []

for file in file_ocv_list:
    with open(file, "r", encoding="latin-1") as f:
        number2index = int(re.findall(r"\d+", str(f.readlines(50)))[0])
        # print(number2index)  # noqa: ERA001
        for line_index, line in enumerate(f, start=1):  # noqa: B007
            if line_index == number2index - 2:
                # print(line.rstrip('\n'), type(line))  # noqa: ERA001
                break
        name = line.split()

        df = pd.read_csv(
            file,
            sep=r"\t",
            index_col=False,
            header=None,
            names=name,
            skiprows=number2index,
            encoding="latin-1",
            parse_dates=[
                name[0],
            ],
            engine="python",
        )
        df_ocv.append(df)

for file in file_gcpl_list:
    with open(file, "r", encoding="latin-1") as f:
        number2index = int(re.findall(r"\d+", str(f.readlines(50)))[0])
        # print(number2index)  # noqa: ERA001
        for line_index, line in enumerate(f, start=1):  # noqa: B007
            if line_index == number2index - 2:
                # print(line.rstrip('\n'), type(line))  # noqa: ERA001
                break
        name = line.split()
        df = pd.read_csv(
            file,
            sep=r"\t",
            index_col=False,
            header=None,
            names=name,
            skiprows=number2index,
            encoding="latin-1",
            parse_dates=[
                name[0],
            ],
            engine="python",
        )
        df_gcpl.append(df)

for file in file_ca_list:
    with open(file, "r", encoding="latin-1") as f:
        number2index = int(re.findall(r"\d+", str(f.readlines(50)))[0])
        # print(number2index)  # noqa: ERA001
        for line_index, line in enumerate(f, start=1):  # noqa: B007
            if line_index == number2index - 2:
                # print(line.rstrip('\n'), type(line))  # noqa: ERA001
                break
        name = line.split()
        df = pd.read_csv(
            file,
            sep=r"\t",
            index_col=False,
            header=None,
            names=name,
            skiprows=number2index,
            encoding="latin-1",
            parse_dates=[
                name[0],
            ],
            engine="python",
        )
        df_ca.append(df)

df_ocv = pd.concat(
    df_ocv,
    axis=0,
    ignore_index=True,
)
df_gcpl = pd.concat(
    df_gcpl,
    axis=0,
    ignore_index=True,
)
df_ca = pd.concat(
    df_ca,
    axis=0,
    ignore_index=True,
)

df_ca["(Q-Qo)/C"] = df_ca["(Q-Qo)/C"] / 3.6
df_ca.rename(columns={"<Ece>/V": "Ece/V", "(Q-Qo)/C": "(Q-Qo)/mA.h"}, inplace=True)
data = pd.concat([df_ocv, df_gcpl, df_ca], ignore_index=True, axis=0).fillna(0)
data.sort_values(by="time/s", inplace=True, ignore_index=True)
data["timegap/s"] = (data["time/s"] - data["time/s"][0]).dt.total_seconds()

In [ ]:
# 画图
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(7, 5))
gs = gridspec.GridSpec(2, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

image = plt.imread(Path.joinpath(image_path, r"Preparation2.tif"))
ax.imshow(image)
ax.set_axis_off()

scalebar = add_sizebar(ax=ax, size=100, scale=0.22, unit=r"$\mathrm{\mu m}$", color="k")
scalebar.set_bbox_to_anchor((0.0, 0.05), transform=ax.transAxes)  # type: ignore

ax.text(
    -0.03,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

(p1,) = ax.plot(data["timegap/s"] / 60, data["Ece/V"], ls="-", lw=1.0, c=colors[0], label=r"WE")
ax.set_xlim(0, 150)
ax.set_xlabel(r"Time (minutes)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=30))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=15))
ax.set_ylim(-1.0, 2.0)
ax.set_ylabel(r"WE, Potential (V)", fontsize=9, color=p1.get_color())
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0.0))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

ax1 = ax.twinx()
ax1.set_position((0.5, 0.0, 1.0, 1.0))
ax1.set_box_aspect(0.8)
(p2,) = ax1.plot(data["timegap/s"] / 60, data["(Q-Qo)/mA.h"] * 100000, ls="-", lw=1.0, c=colors[1], label=r"Capacity")  # type: ignore
ax1.set_ylim(-0.01, 0.6)
ax1.set_ylabel(r"Capactiy (nA.h)", fontsize=9, color=p2.get_color())
ax1.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax1.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0.0))
ax1.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=False,
    left=False,
    right=True,
    labelbottom=False,
    labelleft=False,
    labeltop=False,
    labelright=True,
)

# ax2 = ax.twinx() # noqa: ERA001
# ax2.set_position([0.5, 0.0, 1.0, 1.0])  # noqa: ERA001
# ax2.set_box_aspect(0.8)  # noqa: ERA001
# ax2.spines.right.set_position(("axes", 1.35)) # noqa: ERA001
# p3, = ax2.plot(data['timegap/s']/60, data['<I>/mA']*1000, ls='-', lw=1.0, c=colors[3], label=r"Current") # noqa: ERA001
# ax2.set_ylim(0, 0.3) # noqa: ERA001
# ax2.set_ylabel(r"Current ($\mathrm{\mu A}$)", fontsize=9, color=p3.get_color()) # noqa: ERA001
# ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0.0)) # noqa: ERA001
# ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0.0)) # noqa: ERA001
# ax2.tick_params(axis='both', which='both', labelsize=9, bottom=False, top=False, left=False, right=True, labelbottom=False, labelleft=False, labeltop=False, labelright=True)  # noqa: E501, ERA001

# ax3 = ax.twinx() # noqa: ERA001
# ax3.set_position([0.5, 0.0, 1.0, 1.0]) # noqa: ERA001
# ax3.set_box_aspect(0.8) # noqa: ERA001
# ax3.spines.right.set_position(("axes", 1.7)) # noqa: ERA001
# p4, = ax3.plot(data['timegap/s']/60, data['Ece/V'], ls='-', lw=1.0, c=colors[3], label=r"CE") # noqa: ERA001
# ax3.set_ylim(-1.0, 2.0) # noqa: ERA001
# ax3.set_ylabel(r"CE_Voltage (V)", fontsize=9, color=p4.get_color()) # noqa: ERA001
# ax3.yaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0)) # noqa: ERA001
# ax3.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0)) # noqa: ERA001
# ax3.tick_params(axis='both', which='both', labelsize=9, bottom=False, top=False, left=False, right=True, labelbottom=False, labelleft=False, labeltop=False, labelright=True)  # noqa: E501, ERA001
# # ax.legend(handles=[p1, p2, p3, p4], loc='upper left', bbox_to_anchor=(0.0, 1), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)  # noqa: E501, ERA001

ax.annotate(
    r"A",
    (60, -0.98),
    xytext=(0.3, 0.18),
    fontsize=9,
    c="k",
    xycoords="data",
    textcoords="axes fraction",
    fontweight="normal",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "k", "ls": "-", "lw": 1.0},
)
ax.annotate(
    r"B",
    (93, -0.35),
    xytext=(0.5, 0.6),
    fontsize=9,
    c="k",
    xycoords="data",
    textcoords="axes fraction",
    fontweight="normal",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "k", "ls": "-", "lw": 1.0},
)
ax.annotate(
    r"C",
    (119, -0.98),
    xytext=(0.95, 0.3),
    fontsize=9,
    c="k",
    xycoords="data",
    textcoords="axes fraction",
    fontweight="normal",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "k", "ls": "-", "lw": 1.0},
)
ax.annotate(
    r"D",
    (126, 1.5),
    xytext=(0.70, 0.8),
    fontsize=9,
    c="k",
    xycoords="data",
    textcoords="axes fraction",
    fontweight="normal",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "k", "ls": "-", "lw": 1.0},
)

ax.text(
    -0.28,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.30, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)
ax.plot(stxm_mn["energy"], stxm_mn["Sol_240310089_Charge"], ls="-", lw=1.0, c=colors[1], label=r"OCV")  # type: ignore
ax.plot(stxm_mn["energy"], stxm_mn["Sol_240310075_Charge"], ls="-", lw=1.0, c=colors[0], label=r"Discharged")  # type: ignore

ax.set_xlim(635, 650)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5))
ax.set_ylim(-0.01, 1.5)
ax.set_ylabel(r"Intensity (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0.0))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="lower right", bbox_to_anchor=(1.0, -0.05), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.28,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.25, 1.0, 1.0))
ax.set_box_aspect(0.8)
image = plt.imread(image_list[0])
ax.imshow(image, cmap="gray")
ax.set_axis_off()
scalebar = add_sizebar(ax=ax, size=5, scale=0.09, unit=r"$\mathrm{\mu m}$", color="k")
scalebar.set_bbox_to_anchor((0.0, 0.05), transform=ax.transAxes)  # type: ignore

ax.annotate(
    r" ",
    (0.25, 0.80),
    xytext=(0.4, 0.95),
    fontsize=9,
    c="w",
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "w", "ls": "-", "lw": 1.0},
)
ax.text(
    0.95,
    0.87,
    "A",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    size=9,
    color="k",
    horizontalalignment="right",
    verticalalignment="center",
)
ax.text(
    -0.03,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.10, 0.25, 1.0, 1.0))
ax.set_box_aspect(0.8)
image = plt.imread(image_list[1])
ax.imshow(image, cmap="gray")
ax.set_axis_off()
scalebar = add_sizebar(ax=ax, size=5, scale=0.09, unit=r"$\mathrm{\mu m}$", color="k")
scalebar.set_bbox_to_anchor((0.0, 0.05), transform=ax.transAxes)  # type: ignore

ax.annotate(
    r" ",
    (0.25, 0.80),
    xytext=(0.4, 0.95),
    fontsize=9,
    c="w",
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "w", "ls": "-", "lw": 1.0},
)
ax.text(
    0.95,
    0.87,
    "B",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    size=9,
    color="k",
    horizontalalignment="right",
    verticalalignment="center",
)
# ax.text(-0.03, 1.0, r'e', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.2, 0.25, 1.0, 1.0))
ax.set_box_aspect(0.8)
image = plt.imread(image_list[2])
ax.imshow(image, cmap="gray")
ax.set_axis_off()
scalebar = add_sizebar(ax=ax, size=5, scale=0.09, unit=r"$\mathrm{\mu m}$", color="k")
scalebar.set_bbox_to_anchor((0.0, 0.05), transform=ax.transAxes)  # type: ignore

ax.annotate(
    r" ",
    (0.25, 0.80),
    xytext=(0.4, 0.95),
    fontsize=9,
    c="w",
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "w", "ls": "-", "lw": 1.0},
)
ax.text(
    0.95,
    0.87,
    "C",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    size=9,
    color="k",
    horizontalalignment="right",
    verticalalignment="center",
)
# ax.text(-0.03, 1.0, r'f', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

# 图 G
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.25, 1.0, 1.0))
ax.set_box_aspect(0.8)
image = plt.imread(image_list[3])
ax.imshow(image, cmap="gray")
ax.set_axis_off()
scalebar = add_sizebar(ax=ax, size=5, scale=0.09, unit=r"$\mathrm{\mu m}$", color="k")
scalebar.set_bbox_to_anchor((0.0, 0.05), transform=ax.transAxes)  # type: ignore

ax.annotate(
    r" ",
    (0.25, 0.80),
    xytext=(0.4, 0.95),
    fontsize=9,
    c="w",
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": "w", "ls": "-", "lw": 1.0},
)
ax.text(
    0.95,
    0.87,
    "D",
    bbox={
        "boxstyle": "Circle, pad=0.4",
        "fc": "w",
        "ec": "k",
    },
    transform=ax.transAxes,
    size=9,
    color="k",
    horizontalalignment="right",
    verticalalignment="center",
)
# ax.text(-0.03, 1.0, r'g', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_23_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_23_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_23_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_23_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 22 XES data 

In [ ]:
# XES 的数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\ExSitu\αMnO2\Kbeta\Results\2023-CLAESS\Results\V2"  # noqa: RUF001
)
xes_peakarea = pd.read_csv(
    Path.joinpath(path_file, r"peak_area_std1.csv"), comment="#", sep=r",", header=0, index_col=None
)
xes_spectrum = pd.read_csv(
    Path.joinpath(path_file, r"all_mean_normal.csv"), comment="#", sep=r",", header=0, index_col=None
)

In [ ]:
xes_peak = xes_peakarea.iloc[2:7, 0:3]  # FD 数据
xes_area = xes_peakarea.iloc[2:7, [0, 5, 6]]  # FD 数据
xes_spectrum = xes_spectrum.iloc[:, [0, 3, 4, 5, 6, 7]]  # FD 数据
diff_spectrum = xes_spectrum.copy(deep=True)
diff_spectrum.iloc[:, 1:] = diff_spectrum.iloc[:, 1:].sub(diff_spectrum.iloc[:, 3], axis=0)

In [ ]:
# 画图
fig = plt.figure(figsize=(7, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [r"$\mathrm{Ref.MnO}$", r"$\mathrm{Ref.Mn_2O_3}$", r"$\mathrm{Ref.MnO_2}$", r"Pristine", r"Discharged"]
lss = [r"--", r"--", r"--", r"-", r"-"]
colormap = ListedColormap(mpl.colormaps["sunset"](np.linspace(0.0, 0.5, xes_spectrum.shape[1] - 1)), name=r"colormap")

# 多线叠加
for i in range(xes_spectrum.shape[1] - 1):
    ax.plot(
        xes_spectrum.iloc[:, 0],
        xes_spectrum.iloc[:, 1 + i],
        lw=1,
        ls=lss[i],
        label=labels[i],
        color=colors[i],  # type: ignore
        zorder=5,
        alpha=1 - 0.01 * i,
    )
    ax.plot(
        xes_spectrum.iloc[:, 0],
        diff_spectrum.iloc[:, 1 + i] - 0.01,
        lw=1,
        ls=lss[i],
        color=colors[i],  # type: ignore
        zorder=5,
        alpha=1 - 0.01 * i,
    )

ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(6465, 6510)
ax.xaxis.set_major_locator(ticker.MultipleLocator(10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(5, offset=0))

ax.set_ylabel(ylabel=r"Relative Intensity (arb.u.)", fontsize=9)
ax.set_ylim(-0.04, 0.16)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.04))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.02))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    columnspacing=0.5,
)

axins = ax.inset_axes((0.68, 0.4, 0.30, 0.55))
for i in range(xes_spectrum.shape[1] - 1):
    axins.plot(
        xes_spectrum.iloc[:, 0],
        xes_spectrum.iloc[:, 1 + i],
        lw=1,
        label=labels[i],
        ls=lss[i],
        color=colors[i],  # type: ignore
        zorder=0,
        alpha=1 - 0.01 * i,
    )
axins.set_xlim(6491, 6495)
axins.set_axis_off()
axins.set(xticks=[], xlabel=None, yticks=[], ylabel=None)
ax.text(
    0.32,
    0.40,
    r"$\mathrm{K}\beta^{\prime}$",
    transform=ax.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.65,
    0.93,
    r"$\mathrm{K}_{\beta 1,3}$",
    transform=ax.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
)

ax.text(
    -0.20,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.2, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.errorbar(
    x=np.arange(len(labels)),
    y=xes_peak.iloc[:, 1],
    yerr=xes_peak.iloc[:, 2],
    c=colors[0],  # type: ignore
    fmt="o",
    linewidth=1,
    capsize=4,
)

ax.set_xticks(np.arange(len(labels)), labels=labels)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)

ax.set_ylabel(r"$\mathrm{K}_{\beta 1,3} \ Position \ (eV)$", fontsize=9, labelpad=10.0)  # Total Magnetization
ax.set_ylim(6492.0, 6493.6)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0))
formatter = ticker.ScalarFormatter(useOffset=6490)
ax.yaxis.set_major_formatter(formatter)

arrowprops = {"arrowstyle": "<-", "color": colors[0], "connectionstyle": "angle,angleA=0,angleB=90,rad=10"}  # type: ignore
ax.annotate(
    r" ", xy=(0.2, 0.5), xycoords="axes fraction", xytext=(0, 0.4), textcoords="axes fraction", arrowprops=arrowprops
)

ax2 = ax.twinx()
ax2.set_position((0.2, 0, 1.0, 1.0))
ax2.set_box_aspect(0.8)

ax2.errorbar(  # type: ignore
    x=np.arange(len(labels)),
    y=xes_area.iloc[:, 1],
    yerr=xes_area.iloc[:, 2],
    c=colors[1],  # type: ignore
    fmt="s",
    linewidth=1,
    capsize=4,
    alpha=0.5,
)

ax2.set_xticks(np.arange(len(labels)), labels=labels)
plt.setp(ax2.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=9)

ax2.set_ylabel(r"Local Magnetic Moment ($\mathrm{\mu _B}$)", fontsize=9)  # Total Magnetization
ax2.set_ylim(2.5, 5.5)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: "%.1f" % x))

arrowprops = {
    "arrowstyle": "<-",
    "color": colors[1],
    "alpha": 0.5,
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
}  # type: ignore
ax2.annotate(  # type: ignore
    r" ", xy=(0.8, 0.6), xycoords="axes fraction", xytext=(1.0, 0.67), textcoords="axes fraction", arrowprops=arrowprops
)

ax.text(
    -0.2, 1.0, r"b", transform=ax.transAxes, fontsize=14, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_22_300.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_22_600.tif"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_22_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_22_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 21 Check EXAFS operando noise

In [ ]:
# 获取数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3"  # noqa: RUF001
)
# EXAFS
EXAFS_Mn = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell3_opXAS_P2a_Mn_Oct2022_Rkbg_1.3_Powder.chir2_mag"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pdf_Mn = EXAFS_Mn.iloc[:, 0:3]  # noqa: N816
pdf_Mn.columns = [r"Distance", r"0.2M_MnSO4(aq.)", r"alpha_MnO2_Electrode"]
opData_Mn = EXAFS_Mn.iloc[:, [0, *list(range(3, 14, 1))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

EXAFS_Zn = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell3_opXAS_P2a_Zn_Oct2022_Rkbg_1.1.chir2_mag"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pdf_Zn = EXAFS_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Distance", r"0.2M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = EXAFS_Zn.iloc[:, [0, *list(range(3, 14, 1))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

In [ ]:
# 画图
fig = plt.figure(figsize=(5.2, 5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0, figure=fig)

# 标样
xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]  # type: ignore
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]
xas_colors = [
    ListedColormap(mpl.colormaps["coolwarm"](np.linspace(0.5, 1.0, opData_Mn.shape[1] - 1)), name="xas_cmap_Mn"),
    ListedColormap(mpl.colormaps["coolwarm"](np.linspace(0.5, 0.0, opData_Zn.shape[1] - 1)), name="xas_cmap_Zn"),
]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(2):
    ax.plot(pdf_Mn.iloc[:, 0], pdf_Mn.iloc[:, 1 + i], ls="--", lw=1.0, c=xascolors[0][i], label=labels[0][i])
ax.legend(loc="upper left", fontsize=10, ncol=1, bbox_to_anchor=(0.5, 1.0), handlelength=1)
for i in range(opData_Mn.shape[1] - 1):
    ax.plot(opData_Mn.iloc[:, 0], opData_Mn.iloc[:, i + 1], c=xas_colors[0].colors[i], ls="-", lw=1.0)  # type: ignore

ax.axvline(x=2.40, ymin=0.0, ymax=0.8, ls="--", color="k", alpha=1.0)
ax.axvline(x=3.01, ymin=0.0, ymax=0.3, ls="--", color="k", alpha=1.0)

ax.set_xlim(0, 10)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))

ax.set_ylim(0, 2.0)
ax.set_ylabel(r"FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.5, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.57, 0.15, 0.40, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors[0],
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False)  # type: ignore

ax.text(
    0.57,
    0.3,
    "OCV",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.73,
    0.3,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(2):
    ax.plot(pdf_Zn.iloc[:, 0], pdf_Zn.iloc[:, 1 + i], ls="--", lw=1.0, c=xascolors[1][i], label=labels[1][i])
ax.legend(loc="upper left", fontsize=10, ncol=1, bbox_to_anchor=(0.5, 1.0), handlelength=1)
for i in range(opData_Zn.shape[1] - 1):
    ax.plot(opData_Zn.iloc[:, 0], opData_Zn.iloc[:, i + 1], c=xas_colors[1].colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(0, 10)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))

ax.set_ylim(0, 2.0)
ax.set_ylabel(r"FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.5, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.57, 0.15, 0.40, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors[1],
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False)  # type: ignore

ax.text(
    0.57,
    0.30,
    "OCV",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.73,
    0.30,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
)

ax.text(
    -0.21,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_21_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_21_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_21_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_21_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 19  EXAFS Simulations: Mn-Mn reatio

In [ ]:
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\FD1st\Oct2022_cell3_P2a\EXAFS\Mn\Simulations\Vlad"  # noqa: E501
)

Mn_pristine_k = pd.read_excel(
    Path.joinpath(path_file, r"extr_Merge_alpha_MnO2_powder2_table.xlsx"),
    sheet_name=2,
    engine="openpyxl",
    comment="#",
    index_col=None,
    header=0,
)
Mn_pristine_r = pd.read_excel(
    Path.joinpath(path_file, r"extr_Merge_alpha_MnO2_powder2_table.xlsx"),
    sheet_name=3,
    engine="openpyxl",
    comment="#",
    index_col=None,
    header=0,
)

Mn_ocv_k = pd.read_excel(
    Path.joinpath(path_file, r"extr_Mn_P2a_00_OCV_table.xlsx"),
    sheet_name=2,
    engine="openpyxl",
    comment="#",
    index_col=None,
    header=0,
)
Mn_ocv_r = pd.read_excel(
    Path.joinpath(path_file, r"extr_Mn_P2a_00_OCV_table.xlsx"),
    sheet_name=3,
    engine="openpyxl",
    comment="#",
    index_col=None,
    header=0,
)

Mn_discharge_k = pd.read_excel(
    Path.joinpath(path_file, r"extr_Copy_1_of_Mn_P2a_10_1st_discharged_097V_table.xlsx"),
    sheet_name=2,
    engine="openpyxl",
    comment="#",
    index_col=None,
    header=0,
)
Mn_discharge_r = pd.read_excel(
    Path.joinpath(path_file, r"extr_Copy_1_of_Mn_P2a_10_1st_discharged_097V_table.xlsx"),
    sheet_name=3,
    engine="openpyxl",
    comment="#",
    index_col=None,
    header=0,
)

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    Mn_pristine_r.iloc[:, 0],
    Mn_pristine_r.iloc[:, 1],
    label=r"Pristine",
    lw=0,
    c=colors[0],  # type: ignore
    marker="o",
    markeredgecolor=colors[0],  # type: ignore
    markerfacecolor="w",
    zorder=0,
)
ax.plot(Mn_pristine_r.iloc[:, 0], Mn_pristine_r.iloc[:, 2], label=r"Fit", c="k", zorder=1)

ax.set_xlim(0, 6)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))

ax.set_ylim(0.0, 2.4)
ax.set_ylabel(r"Relative FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.58, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=0.3,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    -0.28,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    Mn_pristine_k.iloc[:, 0],
    Mn_pristine_k.iloc[:, 1],
    label=r"Pristine",
    lw=0,
    c=colors[0],  # type: ignore
    marker="o",
    markeredgecolor=colors[0],  # type: ignore
    markerfacecolor="w",
    zorder=0,
)
ax.plot(Mn_pristine_k.iloc[:, 0], Mn_pristine_k.iloc[:, 2], label=r"Fit", c="k", zorder=1)

ax.set_xlim(0, 15)
ax.set_xlabel(r"Wavenumber ($\mathrm{\AA ^{-1}}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=3))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5))

ax.set_ylim(-3.5, 2.5)
ax.set_ylabel(r"Relative [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1.2, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.6, offset=0.1))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.58, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=0.3,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    0.95,
    0.07,
    r"Fit Range: 3.0 - 14.0",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
)

ax.text(
    -0.28,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.45, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    Mn_ocv_r.iloc[:, 0],
    Mn_ocv_r.iloc[:, 1],
    label=r"OCV",
    lw=0,
    c=colors[1],  # type: ignore
    marker="o",
    markeredgecolor=colors[1],  # type: ignore
    markerfacecolor="w",
    zorder=0,
)
ax.plot(Mn_ocv_r.iloc[:, 0], Mn_ocv_r.iloc[:, 2], label=r"Fit", c="k", zorder=1)

ax.set_xlim(0, 6)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))

ax.set_ylim(0.0, 1.6)
ax.set_ylabel(r"Relative FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.58, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=0.3,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    -0.28,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.45, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    Mn_ocv_k.iloc[:, 0],
    Mn_ocv_k.iloc[:, 1],
    label=r"OCV",
    lw=0,
    c=colors[1],  # type: ignore
    marker="o",
    markeredgecolor=colors[1],  # type: ignore
    markerfacecolor="w",
    zorder=0,
)
ax.plot(Mn_ocv_k.iloc[:, 0], Mn_ocv_k.iloc[:, 2], label=r"Fit", c="k", zorder=1)

ax.set_xlim(0, 15)
ax.set_xlabel(r"Wavenumber ($\mathrm{\AA ^{-1}}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=3))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5))

ax.set_ylim(-2.5, 2.3)
ax.set_ylabel(r"Relative [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.6, offset=-0.1))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.58, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=0.3,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    0.95,
    0.07,
    r"Fit Range: 3.0 - 14.0",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
)

ax.text(
    -0.28,
    1.0,
    r"e",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.9, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    Mn_discharge_r.iloc[:, 0],
    Mn_discharge_r.iloc[:, 1],
    label=r"Discharged",
    lw=0,
    c=colors[3],  # type: ignore
    marker="o",
    markeredgecolor=colors[3],  # type: ignore
    markerfacecolor="w",
    zorder=0,
)
ax.plot(Mn_discharge_r.iloc[:, 0], Mn_discharge_r.iloc[:, 2], label=r"Fit", c="k", zorder=1)

ax.set_xlim(0, 6)
ax.set_xlabel(r"R Space ($\mathrm{\AA}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))

ax.set_ylim(0.0, 0.9)
ax.set_ylabel(r"Relative FT [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.50, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=0.3,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    -0.28,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.9, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(
    Mn_discharge_k.iloc[:, 0],
    Mn_discharge_k.iloc[:, 1],
    label=r"Discharged",
    lw=0,
    c=colors[3],  # type: ignore
    marker="o",
    markeredgecolor=colors[3],  # type: ignore
    markerfacecolor="w",
    zorder=0,
)
ax.plot(Mn_discharge_k.iloc[:, 0], Mn_discharge_k.iloc[:, 2], label=r"Fit", c="k", zorder=1)

ax.set_xlim(0, 15)
ax.set_xlabel(r"Wavenumber ($\mathrm{\AA ^{-1}}$)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=3))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5))

ax.set_ylim(-1.2, 1.3)
ax.set_ylabel(r"Relative [$\mathrm{{\chi}({\kappa})*{\kappa}{^2}}$]", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=-0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=-0.2))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.50, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    labelspacing=0.3,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    0.95,
    0.07,
    r"Fit Range: 2.5 - 13.0",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    -0.28,
    1.0,
    r"f",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_19_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_19_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_19_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_19_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 18 MCR-ALS of Mn on Cathode

In [ ]:
# mcr 的数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\FD1st\Oct2022_cell3_P2a\XANES"  # noqa: E501, RUF001
)
k0_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"MCR", r"P2a_Mn_MCRHM_1reaction_comp_2_A_B.csv"),
    sep=r",",
    comment="#",
    index_col=0,
    header=0,
)
mcr_Mn = xr.open_dataset(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"MCR", r"P2a_Mn_MCRHM_1reaction_comp_2_A_B.NETCDF4"), engine="h5netcdf"
)
k0_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"MCR", r"P2a_Zn_MCRHM_1reaction_comp_2_A_B.csv"),
    sep=r",",
    comment="#",
    index_col=0,
    header=0,
)
mcr_Zn = xr.open_dataset(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"MCR", r"P2a_Zn_MCRHM_1reaction_comp_2_A_B.NETCDF4"), engine="h5netcdf"
)

# opXANES 的数据
path_xas = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\Overview"
)
xas_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_xas, r"cell3_opXAS_P2a_Mn_Oct2022_Rkbg_1.3_Powder.nor"),
    sep=r"\s+",
    comment="#",
    header=None,
    index_col=None,
)
xas_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_xas, r"cell3_opXAS_P2a_Zn_Oct2022_Rkbg_1.1.nor"),
    sep=r"\s+",
    comment="#",
    header=None,
    index_col=None,
)
# 标样的数据
pdf_Mn = xas_Mn.iloc[:, 0:3]  # noqa: N816
pdf_Mn.columns = [r"Energy", r"0.2M_MnSO4(aq.)", r"alpha_MnO2_Electrode"]
pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZSH"]

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

xas_cmap_Zn = ListedColormap(  # noqa: N816
    mpl.colormaps["coolwarm"](np.linspace(0.5, 0.0, mcr_Mn["MCR_HM_recon_Data"].shape[0] - 1)),
    name="xas_cmap_Zn",
)
xas_cmap_Mn = ListedColormap(  # noqa: N816
    mpl.colormaps["coolwarm"](np.linspace(0.5, 1.0, mcr_Zn["MCR_HM_recon_Data"].shape[0] - 1)),
    name="xas_cmap_Mn",
)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(mcr_Mn["pca_energy"], mcr_Mn["MCR_HM_St"].T, label=[r"MCR-ALS", None], c=colors[0], zorder=5)  # type: ignore
ax.plot(pdf_Mn.iloc[:, 0], pdf_Mn.iloc[:, 1:3], ls="--", label=[r"References", None], c="k", zorder=0)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(0, 2.1)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.6, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.15,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(mcr_Mn["MCR_HM_recon_Data"].shape[0] - 1):
    ax.plot(mcr_Mn["pca_energy"], mcr_Mn["MCR_HM_recon_Data"][i, :], c=xas_cmap_Mn.colors[i], ls="-", lw=1.0)  # type: ignore
    ax.plot(mcr_Mn["pca_energy"], mcr_Mn["MCR_HM_residual"][i, :], c=xas_cmap_Mn.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10))
ax.set_ylim(-0.05, 1.5)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.13, 0.45, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_cmap_Mn,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False)  # type: ignore

ax.text(
    0.50,
    0.26,
    "OCV",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.76,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    -0.15,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.2, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(mcr_Zn["pca_energy"], mcr_Zn["MCR_HM_St"].T, label=[r"MCR-ALS", None], c="b", zorder=5)
ax.plot(pdf_Zn.iloc[:, 0], pdf_Zn.iloc[:, 1:3], ls="--", label=[r"References", None], c="k", zorder=0)


ax.set_xlim(9650, 9740)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.6, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.15,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.15, -0.2, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(mcr_Zn["MCR_HM_recon_Data"].shape[0] - 1):
    ax.plot(mcr_Zn["pca_energy"], mcr_Zn["MCR_HM_recon_Data"][i, :], c=xas_cmap_Zn.colors[i], ls="-", lw=1.0)  # type: ignore
    ax.plot(mcr_Zn["pca_energy"], mcr_Zn["MCR_HM_residual"][i, :], c=xas_cmap_Zn.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(9650, 9740)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))
ax.set_ylim(-0.05, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.50, 0.13, 0.45, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_cmap_Zn,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False)  # type: ignore

ax.text(
    0.50,
    0.26,
    "OCV",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.76,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    -0.15,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_18_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_18_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_18_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_18_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 17 XANES and PCA on electrolyte

In [ ]:
# 读取 reference + operando 数据 # 放电数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3"
)
# Mn
xas_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Overview", r"cell3_opXAS_P1_Mn_Oct2022_Rbkg_1.3_powder.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
charge_index = list(range(0, xas_Mn.shape[1] // 2 + 1, 1))
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:3]  # noqa: N816
pdf_Mn.columns = [r"Energy", r"0.2M_MnSO4(aq.)", r"alpha_MnO2_Electrode"]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(3, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

# Zn
xas_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Overview", r"cell3_opXAS_P1_Zn_Oct2022_Rbkg_1.1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
charge_index = list(range(0, xas_Zn.shape[1] // 2 + 1, 1))
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816
pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.2M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\FD1st\Oct2022_cell3_P1\XANES"  # noqa: E501, RUF001
)
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell3_1stFD_opXAS_P1_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=r",", index_col=None, header=0, comment="#"
)
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell3_1stFD_opXAS_P1_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=r",", index_col=None, header=0, comment="#"
)

In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]  # type: ignore
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(0.5, 1.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(pdf_Mn.shape[1] - 1):
    ax.plot(pdf_Mn.iloc[:, 0], pdf_Mn.iloc[:, i + 1], c=xascolors[0][i], ls="--", lw=1.0, label=labels[0][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Mn.shape[1] - 1):
    ax.plot(opData_Mn.iloc[:, 0], opData_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.04, 0.42, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(loc="upper left", bbox_to_anchor=(0.45, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=10)
ax.text(
    0.49,
    0.18,
    r"OCV",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore
ax.text(
    -0.21,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[-i - 1], ls="-", lw=1.0, zorder=i)  # type: ignore  # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"PCA Intensity", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)
ax.text(
    -0.21,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")  # type: ignore

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.set_ylim(-6.0, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    -0.21,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]  # type: ignore
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.5, 0.0, opData_Zn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

for i in range(pdf_Zn.shape[1] - 1):
    ax.plot(pdf_Zn.iloc[:, 0], pdf_Zn.iloc[:, i + 1], c=xascolors[1][i], ls="--", lw=1.0, label=labels[1][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Zn.shape[1] - 1):
    ax.plot(opData_Zn.iloc[:, 0], opData_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.04, 0.42, 0.07)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(loc="upper left", bbox_to_anchor=(0.45, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=10)
ax.text(
    0.49,
    0.18,
    r"OCV",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore
ax.text(
    -0.21,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[-i - 1], ls="-", lw=1.0, zorder=i)  # type: ignore  # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"PCA Intensity", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)
ax.text(
    -0.21,
    1.0,
    r"e",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")  # type: ignore

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"log of Variance", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(
    -0.21,
    1.0,
    r"f",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_17_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_17_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_17_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_17_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 16 Echem of operando cell

In [ ]:
# 电化学上的时间
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell3\Echem"  # noqa: RUF001
    ).glob(r"**\*.txt")
)
# 读取电化学数据
echem = []
for path_file in path_filelist[0:1]:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

# 电化学上的时间
path_filelist1 = list(
    Path(DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(  # noqa: RUF001
        r"LC*.xlsx"
    )
)
for path_file in path_filelist1[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i].iloc[:, 0].isin([0, 1])]
    echem[i] = echem[i][echem[i].iloc[:, 2] >= 0]

In [ ]:
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1))
ax.set_box_aspect(0.8)

labels = [[r"Beamline", None], [r"Lab", None]]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        if i == 0:
            # 找到电压最小值的索引
            idx_min = temp["Ewe/V"].idxmin()
            # 断开最小值前后的曲线
            ax.plot(
                temp.loc[:idx_min, "Capacity/mA.h"] * 1000 / 1.064,
                temp.loc[:idx_min, "Ewe/V"],
                ls="-",
                lw=1.0,
                c=colors[j],  # type: ignore
                label=labels[i][j],
                zorder=0,
            )
            # ax.plot(temp.loc[idx_min+10:, 'Capacity/mA.h']*1000/1.064, temp.loc[idx_min+10:, 'Ewe/V'], ls='-', lw=1.0, c=colors[j], label=None, zorder=0)  # noqa: E501, ERA001
        if i == 1:
            temp = data[data.iloc[:, 0] == idx]
            # 找到电压最小值的索引
            idx_min = temp[temp["Current/uA"] < 0]["Voltage/V"].idxmin()
            # 断开最小值前后的曲线
            ax.plot(
                temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
                temp.loc[: idx_min - 1, "Voltage/V"],
                ls="--",
                lw=1.0,
                c=colors[1],  # type: ignore
                label=labels[i][j],
                zorder=0,
            )
            # ax.plot(temp.loc[idx_min+1:, 'SpeCap/mAh/g'], temp.loc[idx_min+1:, 'Voltage/V'], ls='--', lw=1.0, c=colors[1], label=None, zorder=0)  # noqa: E501, ERA001

# 设置刻度线等格式
ax.set_xlabel(r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)", fontsize=9)
ax.set_xlim(-5, 350)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax.set_ylim(0.8, 1.4)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

ax.text(
    0.01,
    0.15,
    r"0.5 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[0],  # type: ignore
)
ax.text(
    0.01,
    0.23,
    r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[1],  # type: ignore
)

ax.text(0.01, 0.08, r"C/10", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_16_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_16_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_16_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_16_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Fugire_SI 14 Operando XRD

In [ ]:
path_xrd = Path(DATA_ROOT / r"Zn-Mn\PaperUno\XRD\Operando\αMnO2\2022-ICMAB\Results")  # noqa: RUF001
# 读取 alfaMnO2capillary 数据
xrd = pd.read_csv(
    Path.joinpath(path_xrd, r"Ex situ", r"alfaMnO2capillary.uxd"), sep=r"\s+", index_col=None, header=0, comment="#"
)

# 读取 PDF card 的数据
pdf = pd.read_csv(
    Path.joinpath(path_xrd, r"PDFCards", "PDF-00-044-0141-Mo.csv"), sep=r",", index_col=None, header=0, comment="#"
)
pdfMo = pd.read_csv(  # noqa: N816
    Path.joinpath(path_xrd, r"PDFCards", "PDF_1stDischarge.csv"),
    sep=r",",
    index_col=None,
    header=0,
    skiprows=1,
    comment="#",
)

# 读取 spectrum 的数据
spectum_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stDischarge", "Spectum_all.csv"),
    sep=r",",
    index_col=None,
    header=0,
    comment="#",
)

# 电化学上的时间
with open(Path.joinpath(path_xrd, r"Coin1", r"1stDischarge", r"EchemOperando1_c10_1544mg_C02.txt"), "r") as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Nb header lines"):
        line_skip = int(line.split(":")[1].strip())

echem = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stDischarge", r"EchemOperando1_c10_1544mg_C02.txt"),
    sep="\t",
    index_col=None,
    header=0,
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1, 2],
).dropna(axis=1, how="all", inplace=False)
echem["time/s"] = pd.to_datetime(
    echem["time/s"],
)
echem["Ewe/V"] = pd.to_numeric(
    echem["Ewe/V"],
)
echem["<I>/mA"] = pd.to_numeric(
    echem["<I>/mA"],
)
# echem.info()  # noqa: ERA001

# 谱线上的时间
time_spectrum = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stDischarge", r"Time_index_spectrum.csv"),
    sep=",",
    index_col=0,
    header=0,
    comment="#",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1],
)
time_spectrum["Time"] = pd.to_datetime(time_spectrum["Time"])
# time_spectrum.info()  # noqa: ERA001

# 匹配谱线和电化学上的时间
index_spectrum = [abs(echem["time/s"] - t).idxmin() for t in time_spectrum["Time"]]
index_voltage = (
    echem.loc[index_spectrum, [r"Ewe/V", r"<I>/mA"]].reset_index(drop=False).sort_values(by="Ewe/V", ascending=True)
)
index_voltage.reset_index(drop=False, inplace=True)

In [ ]:
# 选择需要的电化学数据以及对应的谱线
index = [0, 5, 10, 14, 17]
selected = (
    index_voltage[index_voltage["level_0"].isin(index)].sort_values(by="level_0", ascending=True).reset_index(drop=True)
)
selected_echem = echem[echem.index <= selected["index"].iloc[-1]]
mapping_spectum = spectum_all[spectum_all["range"] <= selected["level_0"].iloc[-1] + 1].pivot(
    index="2THETA", columns="range", values="Cnt2_D1"
)
FD1st_time = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

In [ ]:
%matplotlib inline

# gridspec inside gridspec
fig = plt.figure(figsize=(7, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 3, 2], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1))
ax.set_box_aspect(2.5)

ax.plot(selected_echem["Ewe/V"], FD1st_time, ls="-", lw=1.0, c=colors[0], label=r"opCoin_1")  # type: ignore
ax.scatter(selected_echem["Ewe/V"].iloc[selected["index"]], FD1st_time.iloc[selected["index"]], c=colors[1])  # type: ignore

# 特殊位置颜色标定
for i, idx in enumerate([index[0], index[-1]]):
    ax.scatter(
        selected_echem["Ewe/V"].iloc[selected["index"][selected["level_0"] == idx]],
        FD1st_time.iloc[selected["index"][selected["level_0"] == idx]],
        c=colors[0],  # type: ignore
        marker="o",
    )

# # 添加索引文本标注
# for i in range(selected.shape[0]):
#     ax.text(selected_echem['Ewe/V'].iloc[selected['index'].iloc[i]]+0.02, FD1st_time.iloc[selected['index'].iloc[i]], str(selected['level_0'].iloc[i]), fontsize=10, verticalalignment='bottom', horizontalalignment='right')  # noqa: E501, ERA001

ax.set_xlabel(
    r"Voltage (V)",
    fontsize=11,
)
ax.set_xlim(0.85, 1.45)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=-0.05))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=-0.05))

ax.set_ylabel(r"Duration Time (hour)", fontsize=11, labelpad=5)
ax.set_ylim(0, 15)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0))

ax2 = ax.twiny()
ax2.set_position((0, 0, 1.0, 1.0))
ax2.set_box_aspect(2.5)

ax2.plot(selected_echem["<I>/mA"], FD1st_time, ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"Current (mA)",
    fontsize=11,
)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=True,
    left=False,
    right=False,
    labelbottom=False,
    labelleft=False,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=11)  # noqa: E501, ERA001
ax.text(
    -0.5, 1.15, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.05, 0, 1, 1))
ax.set_box_aspect(1)

ax.contourf(mapping_spectum.index, mapping_spectum.columns, mapping_spectum.T, cmap="coolwarm", levels=100)

ax.set_yticks([])
ax.set_xlim(5, 20)
ax.set_xlabel(
    r"2-Theta$\mathrm{_{Mo}}$ (degree)",
    fontsize=11,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=-5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-5))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="center",
    fontfamily="Arial",
)

pdf_scale = [0.001, 0.006, 0.001, 0.03, 0.002, 0.001, 0.0001, 0.0005]
XRD_y = [0, 0.15, 0.15, 0.15, 0.15, 0.07, 0.0, 0.09]
XRDcolors = ["blue", "k", "k", "k", "k", "orange", "purple", "green"]
for i in range(pdfMo.shape[0]):
    temp = pdfMo.iloc[:, 2 * i : 2 * i + 2].dropna(axis="index", how="all")
    for j in range((temp.shape[0])):
        ax.axvline(
            x=temp.iloc[j, 0],
            ymin=XRD_y[i],
            ymax=(XRD_y[i] + temp.iloc[j, 1] * pdf_scale[i]),
            lw=1,
            c=XRDcolors[i],  # type: ignore
        )
ax.text(
    -0.05,
    1.14,
    "b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.05, -0.18, 1.4, 0.2))

ax.text(
    0,
    0.55,
    r"$\alpha$-MnO$\mathrm{_2}$ #00-044-0141",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=10,
    c="blue",
)
ax.text(
    0.48,
    0.55,
    r"Zn(HSO$_{4}$)$_{2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=10,
    c="orange",
)
ax.text(
    0, 0.05, r"ZSH", horizontalalignment="left", verticalalignment="bottom", transform=ax.transAxes, fontsize=10, c="k"
)
ax.text(
    0.12,
    0.04,
    r"K$_{0.66}$Mn$_4$O$_{8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=10,
    c="purple",
)
ax.text(
    0.66,
    0.05,
    r"Ti #00-044-1294",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="green",
)
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor("gainsboro")

opXRD_ref_peak = [[5.87], [8.31], [13.15], [17.05], [18.19], [18.98]]  # noqa: N816
opXRD_colors = ["blue", "blue", "blue", "blue", "green", "blue"]  # noqa: N816

for i in range(len(opXRD_ref_peak)):
    ax = subfig.add_axes((0 + i * 0.23, 0.13, 0.2, 0.85), zorder=0)
    temp = mapping_spectum.loc[
        (mapping_spectum.index >= opXRD_ref_peak[i][0] - 0.1) & (mapping_spectum.index <= opXRD_ref_peak[i][0] + 0.1)
    ]
    ax.contourf(temp.index, temp.columns, temp.T, cmap="coolwarm", levels=100)
    ax.axvline(opXRD_ref_peak[i][0], ls="--", lw=1.0, color="k", alpha=0.6)
    ax.yaxis.set_ticks([])
    ax.set_ylim(temp.columns.min(), temp.columns.max())
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator(opXRD_ref_peak[i]))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opXRD_colors[i],
        bottom=True,
        top=False,
        left=True,
        right=False,
        labelbottom=True,
        labelleft=True,
        labeltop=False,
        labelright=False,
    )
    for j in index:
        ax.axhline(y=j + 1, lw=0.6, ls="--")
ax.text(
    -6, 1.2, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_14_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_14_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_14_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_14_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 13 XPS of pristine alpha MnO2 powder: Mn, O, K

In [ ]:
# 读取数据
path_xps = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XPS+UPS\ExSitu\αMnO2\Pristine\Powder\Results\2023-ICN2\V1"
)
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"Mn2p3_2.dat"), sep=r"\s+", skiprows=2, index_col=None, header=1, comment="#"
)
K_xps = pd.read_csv(Path.joinpath(path_xps, "K2p.dat"), sep=r"\s+", skiprows=2, index_col=None, header=1, comment="#")
O_xps = pd.read_csv(Path.joinpath(path_xps, "O1s.dat"), sep=r"\s+", skiprows=2, index_col=None, header=1, comment="#")

In [ ]:
# 数据处理
# Mn
envelope_Mn_IV = Mn_xps.iloc[:, 3:11].sum(axis=1) - (Mn_xps.iloc[:, -4] * 7)  # envelope_Mn_IV  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 11:19].sum(axis=1) - (Mn_xps.iloc[:, -4] * 7)  # envelope_Mn_III  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 19:27].sum(axis=1) - (Mn_xps.iloc[:, -4] * 7)  # envelope_Mn_II  # noqa: N816
# K
if r"Background_K_2p" not in K_xps.columns:
    K_xps.insert(11, "Background_K_2p", K_xps.iloc[:, 7].copy(deep=True))

for i in range(K_xps.shape[1] - 5):
    K_xps.iloc[:, i + 2] -= K_xps.loc[:, "Background_K_2p"]
# K_xps.info()  # noqa: ERA001
# O
if r"Background_O_1s" not in O_xps.columns:
    O_xps.insert(10, "Background_O_1s", O_xps.iloc[:, 6].copy(deep=True))

for i in range(O_xps.shape[1] - 5):
    O_xps.iloc[:, i + 2] -= O_xps.loc[:, "Background_O_1s"]
# O_xps.info()  # noqa: ERA001


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 1], Mn_xps.iloc[:, 2], label=r"Data", color=colors[0])  # type: ignore # each peak of Mn_xps_data
ax.plot(
    Mn_xps.iloc[:, 1],
    Mn_xps.iloc[:, 28],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)  # envelope_all
ax.plot(Mn_xps.iloc[:, 1], Mn_xps.iloc[:, -4], label=r"Background", color=colors[2], zorder=3)  # type: ignore # background

ax.plot(
    Mn_xps.iloc[:, 1],
    envelope_Mn_IV,
    color=colors[3],
    linestyle="--",
    label=r"$\mathrm{Mn^{IV}}$",  # type: ignore
)  # envelope_Mn_IV
ax.plot(
    Mn_xps.iloc[:, 1],
    envelope_Mn_III,
    color=colors[5],
    linestyle="--",
    label=r"$\mathrm{Mn^{III}}$",  # type: ignore
)  # envelope_Mn_III
ax.plot(
    Mn_xps.iloc[:, 1],
    envelope_Mn_II,
    color=colors[4],
    linestyle="--",
    label=r"$\mathrm{Mn^{II}}$",
    zorder=3,  # type: ignore
)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 1], Mn_xps.iloc[:, 3:11], color=colors[3], alpha=0.5)  # type: ignore # each peak of Mn_xps_Mn_IV
for i in range(8):
    ax.fill_between(
        x=Mn_xps.iloc[:, 1],
        y1=Mn_xps.iloc[:, i + 3],
        y2=Mn_xps.iloc[:, -4],
        facecolor=colors[3],
        alpha=0.3,  # type: ignore
    )
ax.plot(Mn_xps.iloc[:, 1], Mn_xps.iloc[:, 11:19], color=colors[5], alpha=0.5)  # type: ignore # each peak of Mn_xps_Mn_III
for i in range(8):
    ax.fill_between(
        x=Mn_xps.iloc[:, 1],
        y1=Mn_xps.iloc[:, i + 11],
        y2=Mn_xps.iloc[:, -4],
        facecolor=colors[5],
        alpha=0.3,  # type: ignore
    )
ax.plot(Mn_xps.iloc[:, 1], Mn_xps.iloc[:, 19:27], color=colors[4], alpha=0.5)  # type: ignore # each peak of Mn_xps_Mn_II
for i in range(8):
    ax.fill_between(
        x=Mn_xps.iloc[:, 1],
        y1=Mn_xps.iloc[:, i + 19],
        y2=Mn_xps.iloc[:, -4],
        facecolor=colors[4],
        alpha=0.3,  # type: ignore
    )

ax.set_xlabel(r"Binding Energy (eV)", fontsize=9)
ax.set_xlim(648.0, 638.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(2000, 17000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3000, offset=2000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1500, offset=2000))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(axis="both", labelsize=9)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.02),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)
ax.text(
    0.03,
    0.70,
    r"$\mathrm{Mn^{III}}$:",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[5],  # type: ignore
)
ax.text(
    0.03,
    0.62,
    r"48.8(3) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[5],  # type: ignore
)
ax.text(
    0.03,
    0.50,
    r"$\mathrm{Mn^{IV}}$:",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[3],  # type: ignore
)
ax.text(
    0.03,
    0.42,
    r"49.7(6) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
    color=colors[3],  # type: ignore
)
ax.text(
    0.72,
    0.87,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=11,
)
ax.text(
    -0.21, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_axes((0.35, 0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 1], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[0], label="Data")  # type: ignore
ax.plot(O_xps.iloc[:, 1], O_xps.iloc[:, 7], ls="--", lw=1.0, c=colors[1], label="Fit")  # type: ignore
ax.plot(O_xps.iloc[:, 1], O_xps.iloc[:, 6], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)  # type: ignore

ax.plot(O_xps.iloc[:, 1], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")  # type: ignore
ax.fill_between(x=O_xps.iloc[:, 1], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 6], facecolor=colors[3], alpha=0.5)  # type: ignore
ax.plot(
    O_xps.iloc[:, 1],
    O_xps.iloc[:, 4],
    ls="-",
    lw=1.0,
    c=colors[4],
    label="Hydroxide, Hydrated\n or Defective Oxide",  # type: ignore
)
ax.fill_between(x=O_xps.iloc[:, 1], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 6], facecolor=colors[4], alpha=0.5)  # type: ignore
ax.plot(O_xps.iloc[:, 1], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[5], label=r"Water")  # type: ignore
ax.fill_between(x=O_xps.iloc[:, 1], y1=O_xps.iloc[:, 5], y2=O_xps.iloc[:, 6], facecolor=colors[5], alpha=0.5)  # type: ignore

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(538.0, 526.0)
ax.set_xlabel("Binding Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))

ax.set_ylim(0, 15000)
ax.set_ylabel("Intensity (counts)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1500))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(axis="both", labelsize=9)

ax.annotate(
    r"529.5(7) eV",
    (0.72, 0.85),
    xytext=(0.30, 0.9),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],  # type: ignore
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},  # type: ignore
)
ax.annotate(
    r"530.8(3) eV",
    (0.58, 0.16),
    xytext=(0.25, 0.28),
    fontsize=9,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},  # type: ignore
)
ax.annotate(
    r"532.8(8) eV",
    (0.40, 0.02),
    xytext=(0.1, 0.12),
    fontsize=9,
    c=colors[5],  # type: ignore
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},  # type: ignore
)
ax.text(
    0.80,
    0.87,
    r"O $\mathit{1s}$",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=11,
)
ax.text(
    -0.21, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_axes((0.7, 0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(K_xps.iloc[:, 1], K_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[0], label="Data")  # type: ignore
ax.plot(K_xps.iloc[:, 1], K_xps.iloc[:, 8], ls="--", lw=1.0, c=colors[1], label="Fit")  # type: ignore
ax.plot(K_xps.iloc[:, 1], K_xps.iloc[:, 7], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)  # type: ignore

ax.plot(K_xps.iloc[:, 1], K_xps.iloc[:, 3:5], ls="-", lw=1.0, c=colors[3], label=[r"K-Mn", None])  # type: ignore
ax.fill_between(
    x=K_xps.iloc[:, 1],
    y1=K_xps.iloc[:, 3],
    y2=K_xps.iloc[:, 7],
    facecolor=colors[3],
    alpha=0.5,
    label=None,  # type: ignore
)
ax.fill_between(
    x=K_xps.iloc[:, 1],
    y1=K_xps.iloc[:, 4],
    y2=K_xps.iloc[:, 7],
    facecolor=colors[3],
    alpha=0.5,
    label=None,  # type: ignore
)
ax.plot(K_xps.iloc[:, 1], K_xps.iloc[:, 5:7], ls="-", lw=1.0, c=colors[4], label=[r"K-O", None])  # type: ignore
ax.fill_between(
    x=K_xps.iloc[:, 1],
    y1=K_xps.iloc[:, 5],
    y2=K_xps.iloc[:, 7],
    facecolor=colors[4],
    alpha=0.5,
    label=None,  # type: ignore
)
ax.fill_between(
    x=K_xps.iloc[:, 1],
    y1=K_xps.iloc[:, 6],
    y2=K_xps.iloc[:, 7],
    facecolor=colors[4],
    alpha=0.5,
    label=None,  # type: ignore
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.0,
)
ax.set_xlim(298.0, 290.0)
ax.set_xlabel("Binding Energy (eV)", fontsize=9)
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.0))

ax.set_ylim(0, 2400)
ax.set_ylabel("Intensity (counts)", fontsize=9)
ax.tick_params(axis="x", labelsize=9)
ax.tick_params(axis="y", labelsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=600))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=300))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
ax.yaxis.set_major_formatter(formatter)

ax.text(
    0.80,
    0.87,
    r"K $\mathit{2p}$",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=12,
)

ax.text(
    1.0,
    0.6,
    r"K 2p$\mathit{\mathrm{_{3/2}}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    color=colors[5],  # type: ignore
    fontsize=10,
)
ax.text(
    0.4,
    0.4,
    r"K 2p$\mathit{\mathrm{_{1/2}}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    color=colors[5],  # type: ignore
    fontsize=10,
)

ax.axvline(x=291.82, color="black", ymin=0.67, ymax=0.75, linestyle="--")
ax.axvline(x=294.58, color="black", ymin=0.35, ymax=0.7, linestyle="--")
ax.text(
    0.72,
    0.62,
    r"2.7(7) eV",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    weight="bold",
    fontsize=10,
    c="black",
)
ax.annotate(
    "",
    xy=(0.78, 0.67),
    xytext=(0.42, 0.67),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops={"arrowstyle": "<->", "color": "black", "lw": 1},
    zorder=10,
)
ax.text(
    -0.21, 1.0, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_13_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_13_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_13_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_13_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 12 TEM-EELS of pristine alpha MnO2 powder

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, data, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / data.axes_manager["y"].scale,
        "{} {}".format(size, data.axes_manager["y"].units),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=0.5,
        frameon=False,
        color=color,
        label_top=True,
        fontproperties={"size": 10.0},
    )
    ax.add_artist(asb)


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 读取数据
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2024-EMCA\EELS"  # noqa: RUF001
)
haddf = hs.load(Path.joinpath(path_file, r"Data", r"SI1_150pA_10ms", r"STEM SI.dm4"))
labels = hs.load(
    Path.joinpath(path_file, r"Results", r"SI1_150pA_10ms", r"Hyperspy", r"V1", r"6-CS_cluster_labels_Mn_2.hspy")
)
signal_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Results", r"SI1_150pA_10ms", r"Hyperspy", r"V1", r"Pristine_Mn_Clusters.nor"),
    sep=r"\s+",
    comment="#",
    header=0,
    index_col=None,
)
signal_O = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Results", r"SI1_150pA_10ms", r"Hyperspy", r"V1", r"Pristine_O_Clusters.nor"),
    sep=r"\s+",
    comment="#",
    header=0,
    index_col=None,
)
onsetenergy = hs.load(
    Path.joinpath(path_file, r"Results", r"SI1_150pA_10ms", r"Hyperspy", r"V1", r"5-ps_onset_energy_Mn_L3_b.hspy")
)  # noqa: E501


In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_aspect(1.0)

ax.imshow(haddf[1].data, cmap="gray")
add_sizebar(ax, 10, haddf[1], "w")
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.22, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_axes((0.35, 0, 1, 1), zorder=0)
ax.set_aspect(1.0)

ax.imshow(labels.inav[0].data, cmap="gray")
add_sizebar(ax, 10, labels.inav[0], "w")
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"Bulk Cluster",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.22, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_axes((0.55, 0, 1, 1), zorder=0)
ax.set_aspect(1.0)

ax.imshow(labels.inav[1].data, cmap="gray")
add_sizebar(ax, 10, labels.inav[1], "w")
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"Surface Cluster",
    transform=ax.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.05, 1.0, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_axes((0, 0.1, 1, 1), zorder=0)
ax.set_box_aspect(0.8)
Clusters = ["Bulk", "Surface"]
for i in range(signal_O.shape[1] - 1):
    ax.plot(signal_O.iloc[:, 0], signal_O.iloc[:, i + 1], c=colors[i], ls="-", lw=1.0, label=Clusters[i], zorder=i)  # type: ignore
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.50, 0.30),
    ncols=1,
    frameon=False,
    fontsize=11,
    labelcolor="linecolor",
    columnspacing=0.4,
)
ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=11)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.set_xlim(510.0, 590.0)
ax.set_ylim(-0.03, 1.2)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))

ax.text(0.03, 0.98, r"O $\mathit{K}$ edge", transform=ax.transAxes, fontsize=11, va="top", fontfamily="Arial")
ax.text(
    -0.22, 1.0, "d", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_axes((0.35, 0.1, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

for i in range(signal_Mn.shape[1] - 1):
    ax.plot(signal_Mn.iloc[:, 0], signal_Mn.iloc[:, i + 1], c=colors[i], ls="-", lw=1.0, label=Clusters[i], zorder=i)  # type: ignore

ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=11)
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.set_xlim(620.0, 700.0)
ax.set_ylim(-0.03, 2.8)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.7, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.35, offset=0))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.50, 0.30),
    ncols=1,
    frameon=False,
    fontsize=11,
    labelcolor="linecolor",
    columnspacing=0.4,
)
ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(0.03, 0.98, r"Mn $\mathit{L}$ edge", transform=ax.transAxes, fontsize=11, va="top", fontfamily="Arial")
ax.text(
    -0.22, 1.0, "e", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.50, 0.55, 0.45, 0.4), ylim=(1.0, 2.5), xlim=(637.0, 645.0))

for i in range(signal_Mn.shape[1] - 1):
    axins.plot(signal_Mn.iloc[:, 0], signal_Mn.iloc[:, i + 1], c=colors[i], ls="-", lw=1.0, label=Clusters[i], zorder=i)  # type: ignore

axins.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=-1))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=-1))

axins.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=False,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=False,
    labelright=False,
)
axins.text(
    0.9,
    0.8,
    r"$\mathrm{L_3}$",
    transform=axins.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((0.55, 0.1, 1, 1), zorder=0)
ax.set_box_aspect(0.8)
vmin_L3 = onsetenergy.nanmin().data[0]  # noqa: N816
vmax_L3 = onsetenergy.nanmax().data[0]  # noqa: N816
N_color = 5
# 创建 colormap 并设置 over/under 颜色
cmap = plt.cm.hot_r(np.linspace(0.2, 0.8, 256))  # type: ignore
cmap = LinearSegmentedColormap.from_list("cmap", cmap)
cmap.set_over("white")  # 超出 vmax 显示白色
cmap.set_under("black")  # 低于 vmin 显示黑色

onsetenergy_a = np.nan_to_num(onsetenergy.data, copy=True, nan=0)
# 设置 norm
norm = Normalize(vmin=vmin_L3, vmax=vmax_L3)

ima = ax.imshow(np.flipud(onsetenergy_a), cmap=cmap, norm=norm)
add_sizebar(ax, 10, onsetenergy, "k")
ax.set_axis_off()
ax.tick_params(
    axis="both",
    which="both",
    bottom=False,
    top=False,
    left=False,
    labelbottom=False,
    labelleft=False,
)
cax = subfig.add_subplot()
cax.set_position((1.59, 0.23, 0.07, 0.6))
subfig.colorbar(
    mappable=ima,
    cax=cax,
    ticks=np.linspace(vmin_L3, vmax_L3, N_color),
    format="{x:.1f}",
    location="right",
    orientation="vertical",
)
cax.tick_params(axis="x", direction="out")
cax.text(
    -0.02,
    1.2,
    r"Energy (eV)",
    horizontalalignment="left",
    verticalalignment="top",
    transform=cax.transAxes,
    fontsize=10,
    c="k",
    rotation="horizontal",
)
ax.text(
    -0.05, 1.0, "f", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_12_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_12_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_12_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_12_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 11 TEM-EDX sum fitting of pristine alpha MnO2 powder

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, data, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / data.axes_manager["y"].scale,
        "{} {}".format(size, data.axes_manager["y"].units),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
        fontproperties={"size": 8.0},
    )
    ax.add_artist(asb)


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 读取数据
path_file1 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2\Powders\2023-EMCA\EDS\Results\0007 - B2 HAADF"  # noqa: E501, RUF001
)
path_file2 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2\Powders\2023-EMCA\EDS\Results\0008 - B2 HAADF"  # noqa: E501, RUF001
)
path_file3 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2023-EMCA\EDS\Results\0004 - 1214 Pristine MnO2 HAADF 58000 x"  # noqa: E501, RUF001
)
path_file4 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2023-EMCA\EDS\Results\0005 - 1215 Pristine MnO2 HAADF 58000 x"  # noqa: E501, RUF001
)
path_file5 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2023-EMCA\EDS\Results\0016 - 1326 Pristine MnO2 HAADF 5100 x"  # noqa: E501, RUF001
)
path_file6 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2024-EMCA\EDS\Results\0002 - B8_HAADF_47000_x"  # noqa: E501, RUF001
)

file1 = xr.open_dataset(Path.joinpath(path_file1, r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"), engine="h5netcdf")
file2 = xr.open_dataset(Path.joinpath(path_file2, r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"), engine="h5netcdf")
file3 = xr.open_dataset(Path.joinpath(path_file3, r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"), engine="h5netcdf")
file4 = xr.open_dataset(Path.joinpath(path_file4, r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"), engine="h5netcdf")
file5 = xr.open_dataset(Path.joinpath(path_file5, r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"), engine="h5netcdf")
file6 = xr.open_dataset(Path.joinpath(path_file6, r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"), engine="h5netcdf")

path_file1 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2\Powders\2023-EMCA\EDS\Data\0007 - B2 HAADF"  # noqa: E501, RUF001
)
path_file2 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2\Powders\2023-EMCA\EDS\Data\0008 - B2 HAADF"  # noqa: E501, RUF001
)
path_file3 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2023-EMCA\EDS\Data\0004 - 1214 Pristine MnO2 HAADF 58000 x"  # noqa: E501, RUF001
)
path_file4 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2023-EMCA\EDS\Data\0005 - 1215 Pristine MnO2 HAADF 58000 x"  # noqa: E501, RUF001
)
path_file5 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2023-EMCA\EDS\Data\0016 - 1326 Pristine MnO2 HAADF 5100 x"  # noqa: E501, RUF001
)
path_file6 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2024-EMCA\EDS\Data\0002 - B8_HAADF_47000_x"  # noqa: E501, RUF001
)
haddf1 = hs.load(Path.joinpath(path_file1, r"0007 - B2 HAADF.emd"))
haddf2 = hs.load(Path.joinpath(path_file2, r"0008 - B2 HAADF.emd"))
haddf3 = hs.load(Path.joinpath(path_file3, r"0004 - 1214 Pristine MnO2 HAADF 58000 x.emd"))
haddf4 = hs.load(Path.joinpath(path_file4, r"0005 - 1215 Pristine MnO2 HAADF 58000 x.emd"))
haddf5 = hs.load(Path.joinpath(path_file5, r"0016 - 1326 Pristine MnO2 HAADF 5100 x.emd"))
haddf6 = hs.load(Path.joinpath(path_file6, r"0002 - B8_HAADF_47000_x.emd"))

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file1["Energy"], file1["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file1["Energy"],
    file1["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file1["Energy"], file1["EDS_Sum_Residuals"] - 1500, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-3000, 50000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=10000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=5000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.58, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

peaks = [
    ("C_Ka", 0.273819, 3.0e4),
    ("O_Ka", 0.522233, 3.9e4),
    ("K_Ka", 3.30834, 0.4e4),
    ("Cu_Ka", 8.01638, 0.3e4),
    ("Mn_Ka", 5.87851, 1.6e4),
    ("Mn_Kb", 6.46392, 0.4e4),
    ("Mn_La", 0.644223 + 0.15, 1.4e4),
    # ("P_Ka", 2.00913, 1.5e4), ("S_Ka", 2.30451, 0.4e4), ("Si_Ka", 1.73643, 0.4e4),  # noqa: ERA001
    # ("Zn_Ka", 8.60303, 1.1e4), ("Zn_Kb", 9.53635, 0.3e4), ("Zn_La", 1.01588, 1.7e4),  # noqa: ERA001
]
for name, x, y in peaks:
    ax.text(
        x,
        y,
        name,
        rotation=90,
        verticalalignment="bottom",
        horizontalalignment="center",
        fontsize=9,
        color="k",
        fontfamily="Arial",
        transform=ax.transData,
    )

# 插入图
axins = ax.inset_axes((0.25, 0.28, 0.7, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf1[-2].data, cmap="gray")
add_sizebar(axins, 100, haddf1[-2], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_axes((0.3, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file2["Energy"], file2["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file2["Energy"],
    file2["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file2["Energy"], file2["EDS_Sum_Residuals"] - 1000, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-2000, 25000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.58, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.1, 0.25, 0.7, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf2[-4].data, cmap="gray")
add_sizebar(axins, 100, haddf2[-4], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_axes((0.6, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file3["Energy"], file3["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file3["Energy"],
    file3["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file3["Energy"], file3["EDS_Sum_Residuals"] - 50, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-150, 2000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=500, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=250, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.58, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.1, 0.25, 0.4, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf3[-4].data, cmap="gray")
add_sizebar(axins, 50, haddf3[-4], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file4["Energy"], file4["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file4["Energy"],
    file4["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file4["Energy"], file4["EDS_Sum_Residuals"] - 300, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-600, 10000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.58, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "d", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.1, 0.25, 0.4, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf4[-2].data, cmap="gray")
add_sizebar(axins, 50, haddf4[-2], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_axes((0.3, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file5["Energy"], file5["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file5["Energy"],
    file5["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file5["Energy"], file5["EDS_Sum_Residuals"] - 250, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-500, 8000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.58, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "e", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.1, 0.24, 0.4, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf5[-5].data, cmap="gray")
add_sizebar(axins, 1, haddf5[-5], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((0.6, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file6["Energy"], file6["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file6["Energy"],
    file6["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file6["Energy"], file6["EDS_Sum_Residuals"] - 100, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-200, 3000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.58, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "f", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.1, 0.25, 0.7, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf6[-2].data, cmap="gray")
add_sizebar(axins, 200, haddf6[-2], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_11_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 10 XANES of pristine alpha MnO2 powder: E0 and Pre-Edge

In [ ]:
# 读取数据
path_xas_Mn = Path(  # noqa: N816
    DATA_ROOT / r"Zn-Mn\PaperUno\XAS\ExSitu\αMnO2\XAS\αMnO2\Pristine\Results"  # noqa: RUF001
)
Mn_xas = pd.read_csv(
    Path.joinpath(path_xas_Mn, r"Oxidization state", r"Powder.nor"), sep=r"\s+", index_col=None, header=0, comment="#"
)
Mn_xas2 = pd.read_csv(
    Path.joinpath(path_xas_Mn, r"Oxidization state", r"Powder_E0.dat"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

# XAS
# Linear fitting of Mn oxidation
Mn_xas2_np = Mn_xas2.to_numpy()
A = np.vstack([Mn_xas2_np[:, 1], np.ones(len(Mn_xas2_np[:, 1]))]).T

# 最小二乘解
x, residuals, rank, s = np.linalg.lstsq(A, Mn_xas2_np[:, 0], rcond=None)

# 残差方差估计
m, n = A.shape
sigma2 = residuals[0] / (m - n)

# 协方差矩阵
cov_x = sigma2 * np.linalg.inv(A.T @ A)

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

ax.plot(Mn_xas.iloc[:, 0], Mn_xas.iloc[:, 1], color=colors[4], label=r"MnO")  # type: ignore
ax.plot(Mn_xas.iloc[:, 0], Mn_xas.iloc[:, 2], color=colors[5], label=r"Mn$\mathrm{_2O_3}$")  # type: ignore
ax.plot(Mn_xas.iloc[:, 0], Mn_xas.iloc[:, 3], color=colors[3], label=r"Mn$\mathrm{O_2}$")  # type: ignore
ax.plot(
    Mn_xas.iloc[:, 0],
    Mn_xas.iloc[:, 4],
    color=colors[0],
    linestyle="--",
    label=r"$\mathrm{\alpha}$-MnO$\mathrm{_2}$",  # type: ignore
)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(6520, 6620)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Absorption (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 1.6)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, pos: "%.1f" % y))  # noqa: ARG005
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))

ax.tick_params(axis="both", which="both", labelsize=9)

# 插入图1
# 插图1：拟合与置信带
axins = ax.inset_axes((0.46, 0.14, 0.50, 0.37), xlim=(6537, 6557), ylim=(-0.5, 4.5))

# 数据点
axins.scatter(
    Mn_xas2_np[:, 1],
    Mn_xas2_np[:, 0],
    c=[colors[1], colors[4], colors[5], colors[3]],  # type: ignore
    marker="o",
    label="Ref. Mn(0/II/III/IV)",
)

# α-MnO₂ 点
x_MnO2 = Mn_xas2_np[1, 3]  # noqa: N816
y_MnO2 = x[0] * x_MnO2 + x[1]  # noqa: N816
axins.scatter(x_MnO2, y_MnO2, c=colors[0], marker="*", label=r"$\mathrm{\alpha}$-MnO$\mathrm{_2}$", zorder=6)  # type: ignore

# 拟合线与置信带
x_fit = Mn_xas2_np[:, 1]  # noqa: ERA001
X = np.vstack([x_fit, np.ones_like(x_fit)]).T  # noqa: ERA001
y_fit = X @ x  # noqa: ERA001
stderr = np.sqrt(np.sum(X @ cov_x * X, axis=1))  # noqa: ERA001

axins.plot(x_fit, y_fit, "k--", label="Fit")  # noqa: F821
# axins.fill_between(x_fit, y_fit - stderr, y_fit + stderr, color='gray', alpha=0.3, label='Fit ±1σ')  # noqa: E501, ERA001, RUF003

# 坐标与标签
axins.set_xlabel(r"Energy of $\mathrm{E_0}$ (eV)", fontsize=9, labelpad=-0.1)
axins.set_ylabel(r"Mn Oxidation", fontsize=9)
axins.set_xticks([6537, 6547, 6557])
axins.set_xticklabels([6537, 6547, 6557], fontsize=8)  # type: ignore
axins.set_yticks([0, 1, 2, 3, 4])
axins.set_yticklabels([0, 1, 2, 3, 4], fontsize=8)  # type: ignore
axins.tick_params(which="major", width=1.0, length=2)

handles, labels = axins.get_legend_handles_labels()

# 交换最后两个
handles[-2], handles[-1] = handles[-1], handles[-2]
labels[-2], labels[-1] = labels[-1], labels[-2]
axins.legend(handles, labels, loc="upper left", bbox_to_anchor=(0, 0.52, 0.3, 0.5), frameon=False, fontsize=7)
axins.text(
    0.95,
    0.06,
    r"$\mathrm{\alpha}$-MnO$\mathrm{_2}:3.89\ \pm 0.05$",
    ha="right",
    va="bottom",
    transform=axins.transAxes,
    fontsize=7,
    color=colors[0],  # type: ignore
)

# 插入图 2
axins = ax.inset_axes((0.01, 0.3, 0.23, 0.32), ylim=(-0.01, 0.22), xlim=(6535, 6555))
axins.plot(Mn_xas.iloc[:, 0], Mn_xas.iloc[:, 1], color=colors[4], label=r"MnO")  # type: ignore
axins.plot(Mn_xas.iloc[:, 0], Mn_xas.iloc[:, 2], color=colors[5], label=r"Mn$\mathrm{_2O_3}$")  # type: ignore
axins.plot(Mn_xas.iloc[:, 0], Mn_xas.iloc[:, 3], color=colors[3], label=r"Mn$\mathrm{O_2}$")  # type: ignore
axins.plot(
    Mn_xas.iloc[:, 0],
    Mn_xas.iloc[:, 4],
    color=colors[0],
    linestyle="--",
    label=r"$\mathrm{\alpha}$-MnO$\mathrm{_2}$",  # type: ignore
)
axins.xaxis.set_minor_locator(ticker.LinearLocator(9))

axins.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=False,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=False,
    labelright=False,
)
axins.text(
    0.95,
    0.10,
    r"Pre Edge",
    horizontalalignment="right",
    verticalalignment="bottom",
    transform=axins.transAxes,
    fontsize=8,
    color="k",
)
ax.indicate_inset_zoom(axins, edgecolor="black", ls="--")
# ax.text(-0.14, 1.0, 'a', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_10_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_10_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_10_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_10_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 09 Echem in BulkCell with different electrolytes

In [ ]:
# CP - 1M Na2SO4 + 0.2M MnSO4 - αMnO2 - 1_52mg
# 电化学上的时间
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\PassivatedZnRichCoatting\BulkCell\1M Na2SO4 + 0.2M MnSO4\Results\2024-ICMAB\17-BulkCell-02C-CP-1M Na2SO4+02M MnSO4-aMnO2-1_52mg"  # noqa: E501, RUF001
    ).glob(r"*.txt")
)
# 读取电化学数据
CP_1MNa_02M = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        lines = file.readlines(100)
        for line in lines:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(
        path_file,
        sep=r"\t",
        comment="#",
        skiprows=line_skip - 1,
        encoding="latin_1",
        index_col=None,
        decimal=",",
        engine="python",
    ).dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    CP_1MNa_02M.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(CP_1MNa_02M)):
    CP_1MNa_02M[i] = CP_1MNa_02M[i][CP_1MNa_02M[i].iloc[:, 0].isin([0, 1, 2])]

# CP - 1M ZnSO4 + 0.2M MnSO4 - αMnO2 - 0_8mg
# 电化学上的时间
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\PassivatedZnRichCoatting\BulkCell\1M ZnSO4 + 0.2M MnSO4\Results\2024-ICMAB\16-BulkCell-02C-CP-1M ZnSO4+02M MnSO4-aMnO2-0_8mg"  # noqa: E501, RUF001
    ).glob(r"*.txt")
)
# 读取电化学数据
CP_1MZn_02M = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        lines = file.readlines(100)
        for line in lines:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(
        path_file,
        sep=r"\t",
        comment="#",
        skiprows=line_skip - 1,
        encoding="latin_1",
        index_col=None,
        decimal=".",
        engine="python",
    ).dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    CP_1MZn_02M.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(CP_1MZn_02M)):
    CP_1MZn_02M[i] = CP_1MZn_02M[i][CP_1MZn_02M[i].iloc[:, 0].isin([0, 1, 2])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

for data in CP_1MZn_02M[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        temp = temp[temp["Ewe/V"] > -0.1]  # noqa: PLR2004
        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[:idx_min, "Capacity/mA.h"] * 1000 / 0.8,
            temp.loc[:idx_min, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 40 :, "Capacity/mA.h"] * 1000 / 0.8,
            temp.loc[idx_min + 40 :, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )

for data in CP_1MNa_02M[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # temp = temp[temp['Ewe/V'] > -0.02]  # noqa: ERA001

        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[:idx_min, "Capacity/mA.h"] * 1000 / 1.52,
            temp.loc[:idx_min, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 40 :, "Capacity/mA.h"] * 1000 / 1.52,
            temp.loc[idx_min + 40 :, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlabel(r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)", fontsize=9)
ax.set_xlim(-3, 250)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25))

ax.set_ylabel(r"WE, Voltage (V vs. Ag/AgCl)", fontsize=9)
ax.set_ylim(-0.6, 1.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.2))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)  # noqa: E501, ERA001

ax.text(
    0.03,
    0.21,
    r"1 M ZnSO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[0],  # type: ignore
)
ax.text(
    0.03,
    0.15,
    r"1 M Na$\mathrm{_2}$SO$\mathrm{_4}$ + 0.2 M MnSO$\mathrm{_4}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c=colors[1],  # type: ignore
)
ax.text(0.04, 0.08, r"Bulk Cell, C/5", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")
ax.text(0.2, 0.7, r"pH: 3.66 to 3.81", ha="left", va="top", transform=ax.transAxes, fontsize=10, c=colors[0])  # type: ignore
ax.text(0.2, 0.35, r"pH: 7.23 to 5.36", ha="left", va="top", transform=ax.transAxes, fontsize=10, c=colors[1])  # type: ignore
arrowprops = {
    "arrowstyle": "<-",
    "color": "k",
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "-",
    "alpha": 0.5,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.2, 0.6), xycoords="axes fraction", xytext=(0, 0.5), textcoords="axes fraction", arrowprops=arrowprops
)

# 右边
ax2 = ax.twinx()
ax2.set_position((0, 0, 1.0, 1.0))
ax2.set_box_aspect(0.8)

for data in CP_1MZn_02M[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        temp = temp[temp["Ewe/V"] > -0.1]  # noqa: PLR2004
        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax2.plot(
            temp.loc[:idx_min, "Capacity/mA.h"] * 1000 / 0.8,
            temp.loc[:idx_min, "Ece/V"],
            ls="--",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )
        ax2.plot(
            temp.loc[idx_min + 40 :, "Capacity/mA.h"] * 1000 / 0.8,
            temp.loc[idx_min + 40 :, "Ece/V"],
            ls="--",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )

for data in CP_1MNa_02M[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # temp = temp[temp['Ewe/V'] > -0.02]  # noqa: ERA001
        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax2.plot(
            temp.loc[:idx_min, "Capacity/mA.h"] * 1000 / 1.52,
            temp.loc[:idx_min, "Ece/V"],
            ls="--",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )
        ax2.plot(
            temp.loc[idx_min + 40 :, "Capacity/mA.h"] * 1000 / 1.52,
            temp.loc[idx_min + 40 :, "Ece/V"],
            ls="--",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )

ax2.set_ylabel(r"CE, Voltage (V vs. Ag/AgCl)", fontsize=9)
ax2.set_ylim(-0.6, 1.0)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.2))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.2))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=False,
    top=False,
    left=False,
    right=True,
    labelbottom=False,
    labelleft=False,
    labeltop=False,
    labelright=True,
)
arrowprops = {
    "arrowstyle": "<-",
    "color": "k",
    "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
    "ls": "--",
    "alpha": 0.5,
}  # noqa: E501
ax.annotate(
    r" ", xy=(0.8, 0.8), xycoords="axes fraction", xytext=(1.0, 0.7), textcoords="axes fraction", arrowprops=arrowprops
)
# ax.text(-0.20, 1.0, 'a', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')  # noqa: E501, ERA001

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_09_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_09_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_09_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_09_600.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 08 TEM-EDX sum fitting of the Discharged

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, data, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / data.axes_manager["y"].scale,
        "{} {}".format(size, data.axes_manager["y"].units),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
        fontproperties={"size": 8.0},
    )
    ax.add_artist(asb)


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 读取数据
path_file1 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2023-EMCA\EDS"  # noqa: E501, RUF001
)
path_file2 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS"  # noqa: E501, RUF001
)

file1 = xr.open_dataset(
    Path.joinpath(path_file1, r"Results", r"0009 - A1 HAADF", r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"),
    engine="h5netcdf",  # noqa: E501
)
file2 = xr.open_dataset(
    Path.joinpath(path_file1, r"Results", r"0011 - A1 HAADF", r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"),
    engine="h5netcdf",  # noqa: E501
)
file3 = xr.open_dataset(
    Path.joinpath(path_file2, r"Results", r"0002 - B8_HAADF_47000_x", r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"),
    engine="h5netcdf",  # noqa: E501
)
file4 = xr.open_dataset(
    Path.joinpath(path_file2, r"Results", r"0003 - B8_HAADF_67000_x", r"Sum", r"Hyperspy", r"EDS_Fit_Sum.NETCDF4"),
    engine="h5netcdf",  # noqa: E501
)
file5 = xr.open_dataset(
    Path.joinpath(path_file2, r"Results", r"0005 - B8_HAADF_135_kx", r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"),
    engine="h5netcdf",  # noqa: E501
)
file6 = xr.open_dataset(
    Path.joinpath(path_file2, r"Results", r"0036 - B8_HAADF_135_kx", r"Sum", r"Hyperspy", r"EDS_Sum.NETCDF4"),
    engine="h5netcdf",  # noqa: E501
)

haddf1 = hs.load(Path.joinpath(path_file1, r"Data", r"0009 - A1 HAADF", r"0009 - A1 HAADF.emd"))
haddf2 = hs.load(Path.joinpath(path_file1, r"Data", r"0011 - A1 HAADF", r"0011 - A1 HAADF.emd"))
haddf3 = hs.load(Path.joinpath(path_file2, r"Data", r"0002 - B8_HAADF_47000_x", r"0002 - B8_HAADF_47000_x.emd"))
haddf4 = hs.load(Path.joinpath(path_file2, r"Data", r"0003 - B8_HAADF_67000_x", r"0003 - B8_HAADF_67000_x.emd"))
haddf5 = hs.load(Path.joinpath(path_file2, r"Data", r"0005 - B8_HAADF_135_kx", r"0005 - B8_HAADF_135_kx.emd"))
haddf6 = hs.load(Path.joinpath(path_file2, r"Data", r"0036 - B8_HAADF_135_kx", r"0036 - B8_HAADF_135_kx.emd"))
haddf1[-4].axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file1["Energy"], file1["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file1["Energy"],
    file1["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file1["Energy"], file1["EDS_Sum_Residuals"] - 1500, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-3000, 60000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=10000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=5000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(-0.03, 1.02), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.18,
    1.0,
    "a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

peaks = [
    ("C_Ka", 0.273819, 0.7e4),
    ("O_Ka", 0.522233, 2.4e4),
    ("K_Ka", 3.30834, 0.2e4),
    ("Cu_Ka", 8.01638, 0.4e4),
    ("Mn_Ka", 5.87851, 0.5e4),
    ("Mn_Kb", 6.46392, 0.3e4),
    ("P_Ka", 2.00913, 1.7e4),
    ("S_Ka", 2.30451, 0.4e4),
    ("Si_Ka", 1.73643, 0.4e4),
    ("Zn_Ka", 8.60303, 1.05e4),
    ("Zn_Kb", 9.53635, 0.3e4),
    ("Zn_La", 1.01588, 1.7e4),
]
for name, x, y in peaks:
    ax.text(
        x,
        y,
        name,
        rotation=90,
        verticalalignment="bottom",
        horizontalalignment="center",
        fontsize=9,
        color="k",
        fontfamily="Arial",
        transform=ax.transData,
    )

# 插入图
axins = ax.inset_axes((0.39, 0.27, 0.6, 0.8))
axins.set_aspect(1.0)
axins.imshow(haddf1[-4].data, cmap="gray")
add_sizebar(axins, 500, haddf1[-4], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_axes((0.3, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file2["Energy"], file2["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file2["Energy"],
    file2["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file2["Energy"], file2["EDS_Sum_Residuals"] - 1000, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-2000, 15000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(-0.03, 1.02), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.39, 0.25, 0.6, 0.8))
axins.set_aspect(1.0)
axins.imshow(haddf2[6].data, cmap="gray")
add_sizebar(axins, 200, haddf2[6], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_axes((0.6, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file3["Energy"], file3["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file3["Energy"],
    file3["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file3["Energy"], file3["EDS_Sum_Residuals"] - 1000, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-2000, 20000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.1, 0.75), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.1, 0.32, 0.8, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf3[5].data, cmap="gray")
add_sizebar(axins, 100, haddf3[5], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file4["Energy"], file4["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file4["Energy"],
    file4["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file4["Energy"], file4["EDS_Sum_Residuals"] - 300, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-600, 10000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(-0.03, 1.02), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "d", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.39, 0.25, 0.6, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf4[5].data, cmap="gray")
add_sizebar(axins, 50, haddf4[5], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_axes((0.3, 0, 1, 1), zorder=0)
ax.set_box_aspect(0.8)

ax.plot(file5["Energy"], file5["EDS_Sum_Data"], label=r"Data", color=colors[0])  # type: ignore
ax.plot(
    file5["Energy"],
    file5["EDS_Sum_Fit"],
    label=r"Fit",
    color=colors[1],  # type: ignore
    linestyle="--",
)
ax.plot(file5["Energy"], file5["EDS_Sum_Residuals"] - 1000, label=r"Residual", color=colors[3], zorder=3)  # type: ignore

ax.set_xlabel(r"Energy (keV)", fontsize=9)
ax.set_xlim(-0.01, 10.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-2000, 30000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)

ax.tick_params(
    axis="both",
    which="both",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0.58, 1.02), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    -0.21, 1.0, "e", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 插入图
axins = ax.inset_axes((0.1, 0.24, 0.4, 1.0))
axins.set_aspect(1.0)
axins.imshow(haddf5[-2].data, cmap="gray")
add_sizebar(axins, 100, haddf5[-2], "w")
axins.set_axis_off()
axins.text(
    0.0,
    1.0,
    r"HAADF",
    transform=axins.transAxes,
    fontsize=9,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="k",
)

# # 图 F
# subfig = fig.add_subfigure(gs[1, 2], zorder=0)
# ax = subfig.add_axes((0.6, 0, 1, 1), zorder=0)
# ax.set_box_aspect(0.8)

# ax.plot(file6['Energy'], file6['EDS_Sum_Data'], label=r'Data', color=colors[0])
# ax.plot(file6['Energy'], file6['EDS_Sum_Fit'], label=r'Fit', color=colors[1], linestyle='--', )
# ax.plot(file6['Energy'], file6['EDS_Sum_Residuals']-100, label=r'Residual', color=colors[3], zorder=3)

# ax.set_xlabel(r'Energy (keV)', fontsize=9)
# ax.set_xlim(-0.01, 10.0)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))

# ax.set_ylabel(r'Intensity (counts)', fontsize=9)
# ax.set_ylim(-200, 3000)
# ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000, offset=0))
# ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500, offset=0))
# formatter = ticker.ScalarFormatter()
# formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
# formatter.set_scientific(True)
# ax.yaxis.set_major_formatter(formatter)

# ax.tick_params(axis='both', which='both', bottom=True, top=False, left=True, right=False, labelbottom=True, labelleft=True, labeltop=False, labelright=False)
# ax.legend(loc='upper left', bbox_to_anchor=(0.1, 1.02), ncols=1, frameon=False, labelcolor='linecolor', fontsize=9)
# ax.text(-0.21, 1.0, 'f', transform=ax.transAxes, fontsize=13, va='center', ha='right', fontfamily='Arial', fontweight='bold')

# # 插入图
# axins = ax.inset_axes([0.57, 0.20, 0.4, 1.0])
# axins.set_aspect(1.0)
# axins.imshow(haddf6[-3].data, cmap='gray')
# add_sizebar(axins, 100, haddf6[-3], 'w')
# axins.set_axis_off()
# axins.text(0.0, 1.0, r'HAADF', transform=axins.transAxes, fontsize=9, va='bottom', ha='left', fontfamily='Arial', fontweight='bold', color='k')

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_08_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_08_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_08_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_08_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Fugire_SI 07 Clsuters of discharge Zn

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, scale, unit, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / scale,
        "{} {}".format(size, unit),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
    )
    scalebar = ax.add_artist(asb)
    return scalebar


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


def index(energy, value):
    return np.argmin(abs(energy - value))

In [ ]:
# 读取数据
path_file_zhs = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\STXM\ExSitu\ZHS\Pristine\E5 ZHS\Results\20230630_E6Zn_798.3x826.9y_specnorm\Hyperspy"  # noqa: E501, RUF001
)
zn_zhs = xr.open_dataset(
    Path.joinpath(path_file_zhs, r"20230630_E6Zn_798.3x826.9y_specnorm_Clusters.NETCDF4"), engine="h5netcdf"
)
signal_zhs = pd.read_csv(Path.joinpath(path_file_zhs, r"Zn-ZHS.xmu"), sep=r"\s+", comment="#", header=0, index_col=None)

path_file_1st = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Zn_632.7x1193.9y_specnorm\Hyperspy"  # noqa: E501, RUF001
)
zn_1st = xr.open_dataset(
    Path.joinpath(path_file_1st, r"20230630_B6Zn_632.7x1193.9y_specnorm_Clusters.NETCDF4"), engine="h5netcdf"
)
signal_1st = pd.read_csv(
    Path.joinpath(path_file_1st, r"1stDischarge_Zn.xmu"), sep=r"\s+", comment="#", header=0, index_col=None
)

In [ ]:
clusters0_zhs = zn_zhs["Cluster_Labels"][0].dropna(dim="x_position", how="all")
clusters0_zhs = clusters0_zhs.dropna(dim="y_position", how="all")

clusters1_zhs = zn_zhs["Cluster_Labels"][1].dropna(dim="x_position", how="all")
clusters1_zhs = clusters1_zhs.dropna(dim="y_position", how="all")

clusters0_1st = zn_1st["Cluster_Labels"][0].dropna(dim="x_position", how="all")
clusters0_1st = clusters0_1st.dropna(dim="y_position", how="all")

clusters1_1st = zn_1st["Cluster_Labels"][1].dropna(dim="x_position", how="all")
clusters1_1st = clusters1_1st.dropna(dim="y_position", how="all")

In [ ]:
# 画图

fig = plt.figure(figsize=(7.0, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0, figure=fig)
Clusters = [r"#1", r"#2"]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(clusters0_zhs, cmap="gray")
scalebar = add_sizebar(ax=ax, size=1, scale=0.013000000268220901, unit=r"$\mathrm{\mu m}$", color="w")
scalebar.set_bbox_to_anchor((0, 0.03), transform=ax.transAxes)  # type: ignore
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"#1",
    transform=ax.transAxes,
    fontsize=13,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="y",
)
ax.text(
    -0.03, 1.0, "a", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 A2
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(clusters1_zhs, cmap="gray")
scalebar = add_sizebar(ax=ax, size=1, scale=0.013000000268220901, unit=r"$\mathrm{\mu m}$", color="w")
scalebar.set_bbox_to_anchor((0, 0.03), transform=ax.transAxes)  # type: ignore
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"#2",
    transform=ax.transAxes,
    fontsize=13,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="y",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, 0.07, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(signal_zhs.shape[1] - 1):
    ax.plot(
        signal_zhs.iloc[:, 0],
        signal_zhs.iloc[:, i + 1]
        - signal_zhs.iloc[
            index(signal_zhs.iloc[:, 0].values, 1015) : index(signal_zhs.iloc[:, 0].values, 1025), i + 1
        ].mean(),
        c=colors[i],  # type: ignore
        ls="-",
        lw=1.0,
        label=Clusters[i],
        zorder=i,
    )

ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(1010, 1040)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=9)
ax.set_ylim(-0.03, 0.25)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.05, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.025, offset=0))

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 0.9),
    ncols=1,
    frameon=False,
    fontsize=9,
    labelcolor="linecolor",
    columnspacing=0.4,
)
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(0.03, 0.95, r"Zn $\mathit{L}$ edge", transform=ax.transAxes, fontsize=9, va="top", fontfamily="Arial")
ax.text(
    -0.27, 1.0, "b", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(clusters0_1st, cmap="gray")
scalebar = add_sizebar(ax=ax, size=1, scale=0.013000000268220901, unit=r"$\mathrm{\mu m}$", color="w")
scalebar.set_bbox_to_anchor((0, 0.03), transform=ax.transAxes)  # type: ignore
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"#1",
    transform=ax.transAxes,
    fontsize=13,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="y",
)
ax.text(
    -0.03, 1.0, "c", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

# 图 C2
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

ax.imshow(clusters1_1st, cmap="gray")
scalebar = add_sizebar(ax=ax, size=1, scale=0.013000000268220901, unit=r"$\mathrm{\mu m}$", color="w")
scalebar.set_bbox_to_anchor((0, 0.03), transform=ax.transAxes)  # type: ignore
ax.set_axis_off()
ax.text(
    0.05,
    0.9,
    r"#2",
    transform=ax.transAxes,
    fontsize=13,
    va="bottom",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
    color="y",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, 0.07, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(signal_1st.shape[1] - 1):
    ax.plot(
        signal_1st.iloc[:, 0],
        signal_1st.iloc[:, i + 1]
        - signal_1st.iloc[
            index(signal_1st.iloc[:, 0].values, 1010) : index(signal_1st.iloc[:, 0].values, 1025), i + 1
        ].mean(),
        c=colors[i],  # type: ignore
        ls="-",
        lw=1.0,
        label=Clusters[i],
        zorder=i,
    )

ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=9)
ax.set_ylim(-0.02, 0.25)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.05, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.025, offset=0))

ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(1010, 1040)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 0.9),
    ncols=1,
    frameon=False,
    fontsize=9,
    labelcolor="linecolor",
    columnspacing=0.4,
)
ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True, labelsize=9)

ax.text(0.03, 0.95, r"Zn $\mathit{L}$ edge", transform=ax.transAxes, fontsize=9, va="top", fontfamily="Arial")
ax.text(
    -0.27, 1.0, "d", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold"
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_06_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_06_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_07_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_07_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 06. Surficial Passivated ZnO layer on the α-MnO2 nanowire

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bar, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / bar,
        "{} {}".format(size, "nm"),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=0.5,
        frameon=False,
        color=color,
        label_top=True,
        fontproperties={"size": 10.0},
    )
    ax.add_artist(asb)
    return asb


def add_sizebar1(ax, size, bar, color):
    asb1 = AnchoredSizeBar(
        ax.transData,
        size / bar,
        label=None,
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=0.5,
        frameon=False,
        color=color,
        label_top=True,
        fontproperties={"size": 10.0},
    )
    ax.add_artist(asb1)
    return asb1


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
# 读取数据
eels_1stdischarge = plt.imread(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\TEM\Results\0011 - B8_Ceta_165_kx Ceta.png"  # noqa: E501, RUF001
)
path_eels_zn = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS\Data\SI2_80pA_10ms\Zn.dm4"  # noqa: E501, RUF001
)
path_eels_mn = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS\Results\SI2_80pA_10ms\Hyperspy\V1"  # noqa: E501, RUF001
)
path_eds_1stdischarge = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS\Data\0003 - B8_HAADF_67000_x"  # noqa: E501, RUF001
)

eels_zn = hs.load(path_eels_zn)
eels_mn = hs.load(Path.joinpath(path_eels_mn, r"4-ps_mappings_Mn.hspy"))
onsetenergy_Mn = hs.load(Path.joinpath(path_eels_mn, r"5-ps_onset_energy_Mn_L3_b.hspy")).T
eds_1stdischarge = hs.load(Path.joinpath(path_eds_1stdischarge, r"0003 - B8_HAADF_67000_x.emd"))

In [ ]:
# 混合两张图片
img1 = eds_1stdischarge[8].isig[:, :70].data  # 青蓝色通道
img2 = eds_1stdischarge[6].isig[:, :70].data  # 紫色通道

# 归一化到 0~1（避免超出范围）
img1 = np.clip(img1 / img1.max(), 0, 1)
img2 = np.clip(img2 / img2.max(), 0, 1)

# 分别定义颜色权重
color1 = np.array([0, 245 / 255, 245 / 255])  # 青蓝色
color2 = np.array([252 / 255, 0, 252 / 255])  # 紫色

# 创建 RGB 图像
eds_mapping = np.zeros((*img1.shape, 3))
eds_mapping += img1[..., None] * color1
eds_mapping += img2[..., None] * color2

# 裁剪防止饱和值超过 1
eds_mapping = np.clip(eds_mapping, 0, 1)

In [ ]:
from scipy import ndimage

fig = plt.figure(figsize=(7, 5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0, figure=fig)

# 图 A HAADF
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 0.5, 0.5))
ax.set_box_aspect(1.0)

img = ndimage.rotate(eels_1stdischarge, 0, reshape=True)
ax.imshow(img[3450:3950, 2570:3070], cmap="gray", vmin=0.0, vmax=0.7)
add_sizebar(ax, 5, 0.04028, "k")
ax.set_axis_off()
ax.text(0.05, 0.95, r"HAADF", transform=ax.transAxes, fontsize=13, va="top", ha="left", fontfamily="Arial", c="w")
ax.text(
    -0.08,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 A1 EELS-Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.5, 0, 0.5, 0.5))
ax.set_box_aspect(1.0)

img = eels_mn.data / eels_mn.data.max().max()
# img = ndimage.rotate(img, 1, reshape=True)  # noqa: ERA001
ax.imshow(
    img[70:90, 30:50],
    cmap=transparent_single_color_cmap(color0="k", color1=(236 / 255, 236 / 255, 0)),
    vmin=0.0,
    vmax=1.0,
)

add_sizebar1(ax, 5, eels_mn.axes_manager[0].scale, "w")
ax.set_axis_off()
ax.text(0.05, 0.95, r"Mn", transform=ax.transAxes, fontsize=13, va="top", ha="left", fontfamily="Arial", c="k")

line_Mn = img[70:90, 10:50].mean(  # noqa: N816
    axis=1,
)

# 图 A2 EELS-Zn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((1.0, 0, 0.5, 0.5))
ax.set_box_aspect(1.0)

img = eels_zn.data / eels_zn.data.max().max()
# img = ndimage.rotate(img, 1, reshape=True) # noqa: ERA001
ax.imshow(
    img[70:90, 10:30],
    cmap=transparent_single_color_cmap(color0="k", color1=(252 / 255, 0, 252 / 255)),
    vmin=0.0,
    vmax=0.3,
)
add_sizebar1(ax, 5, eels_zn.axes_manager[0].scale, "w")
ax.set_axis_off()
ax.text(0.05, 0.95, r"Zn", transform=ax.transAxes, fontsize=13, va="top", ha="left", fontfamily="Arial", c="w")

line_Zn = img[70:90, 10:50].mean(  # noqa: N816
    axis=1,
)

# # 图 A3 Onset Energy
# subfig = fig.add_subfigure(gs[0, 2:4], zorder=0) # noqa: ERA001
# ax = subfig.add_subplot() # noqa: ERA001
# ax.set_position([1.6, -0.1, 0.5, 1.0]) # noqa: ERA001
# ax.set_box_aspect(1.0) # noqa: ERA001

# # img = ndimage.rotate(onsetenergy_Mn, 1, reshape=True) # noqa: ERA001
# ax.imshow(onsetenergy_Mn.data[70:90, 10:30], cmap='hot', aspect=1.0,)  # noqa: ERA001
# add_sizebar1(ax, 5, onsetenergy_Mn.axes_manager[0].scale, 'w')  # noqa: ERA001
# ax.set_axis_on()  # noqa: ERA001

line_onsetenergy_Mn = onsetenergy_Mn.data[70:90, 160:-10].mean(  # noqa: N816
    axis=1,
)  # 160:-10
line_onsetenergy_Mn[8] = 635.16
line_onsetenergy_Mn[9] = 635.1

# 图 B
subfig = fig.add_subfigure(gs[0:2, 1:3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.45, 0.21, 0.6, 0.6))
ax.set_box_aspect(0.8)

ax.plot(
    np.linspace(
        0,
        eels_zn.isig[70:90, 10:50].axes_manager["x"].high_value
        - eels_zn.isig[70:90, 10:50].axes_manager["x"].low_value,
        num=line_Zn.shape[0],
    ),
    line_Zn,
    c=colors[1],  # type: ignore
    label="Zn",
)
ax.plot(
    np.linspace(
        0,
        eels_zn.isig[70:90, 10:50].axes_manager["x"].high_value
        - eels_zn.isig[70:90, 10:50].axes_manager["x"].low_value,
        num=line_Mn.shape[0],
    ),
    line_Mn,
    c=colors[0],  # type: ignore
    label="Mn",
)

ax.set_ylabel(r"Relative Intensity (arb.u.)", fontsize=9)
ax.set_ylim(-0.05, 1.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 18)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0))

ax.text(
    0,
    -0.17,
    r"Bulk",
    transform=ax.transAxes,
    fontsize=13,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.83,
    -0.17,
    r"Border",
    transform=ax.transAxes,
    fontsize=13,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.5, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handletextpad=0.3,
    labelspacing=0.3,
)

ax2 = ax.twinx()
ax2.set_position((0.45, 0.21, 0.6, 0.6))
ax2.set_box_aspect(0.8)

ax2.plot(
    np.linspace(
        0,
        eels_zn.isig[70:90, 10:50].axes_manager["x"].high_value
        - eels_zn.isig[70:90, 10:50].axes_manager["x"].low_value,
        num=line_onsetenergy_Mn.shape[0],
    )[0:9],
    line_onsetenergy_Mn[1:10],
    ls="-",
    c=colors[0],  # type: ignore
    label="Onset Energy",
    marker="D",
    markeredgecolor=colors[0],  # type: ignore
    markerfacecolor=colors[0],  # type: ignore
)

ax2.set_ylabel(r"$\mathrm{Mn \ L_3 \ Onset \ Energy (eV)}$", fontsize=9)
ax2.set_ylim(634.8, 635.6)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
formatter = ticker.ScalarFormatter(useOffset=630)
ax2.yaxis.set_major_formatter(formatter)
ax2.legend(
    loc="upper left",
    bbox_to_anchor=(0.5, 0.82),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handletextpad=0.3,
    labelspacing=0.3,
)
ax.text(
    -0.20,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0.2, 1.5, 1.0))
# ax.set_box_aspect(1.0)  # noqa: ERA001

ax.imshow(eds_mapping, alpha=1.0, zorder=2)

# ax.imshow(eds_1stdischarge[8].isig[:, :70].data, cmap=transparent_single_color_cmap(color0='k', color1=(0/255, 245/255, 245/255)), alpha=1.0, vmin=0.0, vmax=0.6, zorder=2) # noqa: E501, ERA001
# ax.imshow(eds_1stdischarge[6].isig[:, :70].data, cmap=transparent_single_color_cmap(color0='k', color1=(252/255, 0, 252/255)), alpha=0.8, vmin=0.0, vmax=3.62, zorder=0)  # noqa: E501, ERA001

add_sizebar(ax, 100, eds_1stdischarge[6].axes_manager[0].scale, "w")
ax.set_axis_off()

ax.text(
    0.03,
    0.9,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": (252 / 255, 0, 252 / 255), "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=13,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    0.125,
    0.9,
    r"S",
    c="w",
    bbox={"ec": None, "fc": (0 / 255, 245 / 255, 245 / 255), "alpha": 0.7, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=13,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    -0.02,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_06_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_06_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_06_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_06_900.svg"),
    transparent=False,
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
)
plt.gcf().set_facecolor("white")
plt.show()

### Fugire_SI 05 TEM-HADDF of Discharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, scale, unit, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / scale,
        "{} {}".format(size, unit),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=1,
        frameon=False,
        color=color,
        label_top=True,
    )
    ax.add_artist(asb)
    return asb

In [ ]:
path_file = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\TEM\Results"  # noqa: E501, RUF001
)
file1 = plt.imread(Path.joinpath(path_file, r"0010 - B8_Ceta_165_kx Ceta.png"))
file6 = plt.imread(Path.joinpath(path_file, r"0015 - B8_Ceta_520_kx Ceta.png"))
file2 = plt.imread(Path.joinpath(path_file, r"0016 - B8_Ceta_265_kx Ceta.png"))
file3 = plt.imread(Path.joinpath(path_file, r"0017 - B8_Ceta_210_kx Ceta.png"))
file4 = plt.imread(Path.joinpath(path_file, r"0018 - B8_Ceta_520_kx Ceta.png"))
file5 = plt.imread(Path.joinpath(path_file, r"0019 - B8_Ceta_520_kx Ceta.png"))

In [ ]:
# 画图
from scipy import ndimage

fig = plt.figure(figsize=(7, 5))
gs = gridspec.GridSpec(
    2,
    3,
    width_ratios=[
        1,
        1,
        1,
    ],
    height_ratios=[1, 1],
    wspace=0.0,
    hspace=0.0,
    figure=fig,
)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = ndimage.rotate(file4, 0, reshape=True)
ax.imshow(img[0:1800, 0:1800, :], cmap="gray", aspect=1.0)
add_sizebar(ax, 10, 0.02376608760343614, r"$\mathrm{\mu m}$", "w")
ax.set_axis_off()


# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.02, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = ndimage.rotate(file5, 0, reshape=True)
ax.imshow(img[0:1800, 0:1800, :], cmap="gray", aspect=1.0)
add_sizebar(ax, 10, 0.02376608760343614, r"$\mathrm{\mu m}$", "w")
ax.set_axis_off()

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.04, 0, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = ndimage.rotate(file3, 0, reshape=True)
ax.imshow(img[400:-850, 400:-850], cmap="gray", aspect=1.0)
add_sizebar(ax, 10, 0.061282566764559225, r"$\mathrm{\mu m}$", "w")
ax.set_axis_off()

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=1)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.05, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = ndimage.rotate(file1, 90, reshape=True)
ax.imshow(img[:-2900, 2900:], cmap="gray", aspect=1.0)
add_sizebar(ax, 10, 0.04028540579939374, r"$\mathrm{\mu m}$", "w")
ax.set_axis_off()

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.02, 0.05, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = ndimage.rotate(file2, 0, reshape=True)
ax.imshow(img[600:-500, 600:-500], cmap="gray", aspect=1.0)
add_sizebar(ax, 10, 0.04685865854604683, r"$\mathrm{\mu m}$", "w")
ax.set_axis_off()

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.04, 0.05, 1.0, 1.0))
ax.set_box_aspect(1.0)

img = ndimage.rotate(file6, 0, reshape=True)
ax.imshow(img[0:1900, 0:1900, :], cmap="gray", aspect=1.0)
add_sizebar(ax, 10, 0.02376608760343614, r"$\mathrm{\mu m}$", "w")
ax.set_axis_off()

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_05_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Fugire_SI 04 Echem in Organic electrolytes

In [ ]:
# 原始 MnO2 在 1M+0.2M 的电化学数据
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\OrganicEchem\StandardElectrolyte\1M ZnSO4 + 0.2M MnSO4\Results"  # noqa: E501, RUF001
    ).glob(r"LC*.xlsx")
)
base_1M_02M = []  # noqa: N816
# 读取电化学数据
for file in path_filelist:
    df = pd.read_excel(file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    base_1M_02M.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(base_1M_02M)):
    if i == 0:
        base_1M_02M[i] = base_1M_02M[i][base_1M_02M[i].iloc[:, 0].isin([1, 3, 4])]
    else:
        base_1M_02M[i] = base_1M_02M[i][base_1M_02M[i].iloc[:, 0].isin([0, 1, 2, 3])]
    base_1M_02M[i] = base_1M_02M[i][base_1M_02M[i].iloc[:, 2] >= 0]
# 调整电话学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
base_1M_02M = [base_1M_02M[i] for i in [1, 0]]  # noqa: N816

# 原始 MnO2 在 138 的电化学数据
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\OrganicEchem\StandardElectrolyte\ZnOTf2-DMSO-H2O (138)\Results"  # noqa: E501, RUF001
    ).glob(r"LC*.xlsx")
)
base_138 = []
# 读取电化学数据
for file in path_filelist:
    df = pd.read_excel(file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    base_138.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(base_138)):
    base_138[i] = base_138[i][base_138[i].iloc[:, 0].isin([0, 1, 2, 3])]
    base_138[i] = base_138[i][base_138[i].iloc[:, 2] >= 0]

# 调整电话学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
base_138 = [base_138[i] for i in [0, 1]]

In [ ]:
# 1st0.9V - ZHS - 1M ZnSO4 + 0.2M MnSO4
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\OrganicEchem\1st0.9V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\28#-04-2023-B"  # noqa: E501, RUF001
    ).glob(r"LC*.xlsx")
)
zhs_1M_02M = []  # noqa: N816
# 读取电化学数据
for file in path_filelist:
    df = pd.read_excel(file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    zhs_1M_02M.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(zhs_1M_02M)):
    zhs_1M_02M[i] = zhs_1M_02M[i][zhs_1M_02M[i].iloc[:, 0].isin([0, 1, 2, 3])]
    zhs_1M_02M[i] = zhs_1M_02M[i][zhs_1M_02M[i].iloc[:, 2] >= 0]

# 调整电话学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
zhs_1M_02M = [zhs_1M_02M[i] for i in [0, 1, 2]]  # noqa: N816

# 1st0.9V - ZHS - 138
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\OrganicEchem\1st0.9V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023"  # noqa: E501, RUF001
    ).glob(r"LC*.xlsx")
)
zhs_138 = []
# 读取电化学数据
for file in path_filelist:
    df = pd.read_excel(file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    zhs_138.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(zhs_138)):
    zhs_138[i] = zhs_138[i][zhs_138[i].iloc[:, 0].isin([0, 1, 2, 3])]
    zhs_138[i] = zhs_138[i][zhs_138[i].iloc[:, 2] >= 0]

# 调整电话学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
zhs_138 = [zhs_138[i] for i in [0, 1, 2]]


In [ ]:
# 1st0.9V - No ZHS - 1M ZnSO4 + 0.2M MnSO4
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\OrganicEchem\1st0.9V\No ZHS\1M ZnSO4 + 0.2M MnSO4\Results\28#-04-2023-B"  # noqa: E501, RUF001
    ).glob(r"LC*.xlsx")
)
no_zhs_1M_02M = []  # noqa: N816
for file in path_filelist:
    df = pd.read_excel(file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    no_zhs_1M_02M.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(no_zhs_1M_02M)):
    no_zhs_1M_02M[i] = no_zhs_1M_02M[i][no_zhs_1M_02M[i].iloc[:, 0].isin([0, 1, 2, 3])]
    no_zhs_1M_02M[i] = no_zhs_1M_02M[i][no_zhs_1M_02M[i].iloc[:, 2] >= 0]

# 调整电话学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
no_zhs_1M_02M = [no_zhs_1M_02M[i] for i in [0, 1, 2]]  # noqa: N816


# 1st0.9V - No ZHS - 138
path_filelist = list(
    Path(
        DATA_ROOT / r"Zn-Mn\PaperUno\Echem\αMnO2\SpecialTest\OrganicEchem\1st0.9V\No ZHS\1M ZnSO4 + 0.2M MnSO4\Results\24#-04-2023-B"  # noqa: E501, RUF001
    ).glob(r"LC*.xlsx")
)
no_zhs_138 = []
for file in path_filelist:
    df = pd.read_excel(file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(
        axis=1, how="all"
    )
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[
        [r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]
    ].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    no_zhs_138.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(no_zhs_138)):
    no_zhs_138[i] = no_zhs_138[i][no_zhs_138[i].iloc[:, 0].isin([0, 1, 2, 3])]
    no_zhs_138[i] = no_zhs_138[i][no_zhs_138[i].iloc[:, 2] >= 0]

# 调整电话学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
no_zhs_138 = [no_zhs_138[i] for i in [1, 2, 0]]

In [ ]:
# 画图
fig = plt.figure(figsize=(7, 10))
gs = gridspec.GridSpec(
    5, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1, 1, 1, 1], wspace=0.25, hspace=0.35, figure=fig
)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in base_138[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )  # labels[j] colors[0]
        ax.plot(
            temp.loc[idx_min + 3 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 3 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )

# 设置刻度线等格式
ax.set_xlim(0, 6)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.98,
    0.1,
    r"$\mathrm{1 \ M \ Zn(OTf){_2} \ in \ DMSO}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.20, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in base_138[1:2]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )  # labels[j] colors[1]
        ax.plot(
            temp.loc[idx_min + 3 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 3 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 150)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=30))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=15))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    1.0,
    0.1,
    r"$\mathrm{Zn(OTf){_2} \ in \ (DMSO + H_{2}O)}$, 1:3:8 wt.%",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.40, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in base_1M_02M[1:2]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[3],  # type: ignore
            label=None,
            zorder=0,
        )  # labels[j] colors[3]
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[3],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 350)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.98,
    0.1,
    r"$\mathrm{1 \ M \ ZnSO{_4} + 0.2 \ M \ MnSO{_4}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.12, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in zhs_1M_02M[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 350)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.98,
    0.1,
    r"$\mathrm{1 \ M \ ZnSO{_4} + 0.2 \ M \ MnSO{_4}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.2, -0.12, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in zhs_1M_02M[1:2]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 2)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.8,
    0.1,
    r"$\mathrm{1 \ M \ Zn(OTf){_2} \ in \ DMSO}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.98,
    0.1,
    r"$\mathrm{ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"e",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, -0.12, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in zhs_1M_02M[2:3]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[3],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[3],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 400)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.8,
    0.1,
    r"$\mathrm{1 \ M \ ZnSO{_4} + 0.2 \ M \ MnSO{_4}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.98,
    0.1,
    r"$\mathrm{ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"f",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 G
subfig = fig.add_subfigure(gs[2, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.24, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in zhs_138[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 350)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.98,
    0.1,
    r"$\mathrm{1 \ M \ ZnSO{_4} + 0.2 \ M \ MnSO{_4}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"g",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 H
subfig = fig.add_subfigure(gs[2, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.2, -0.24, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in zhs_138[1:2]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最大值的索引
        idx_max = temp["Voltage/V"].idxmax()
        # 断开最大值前后的曲线
        ax.plot(
            temp.loc[:idx_max, "SpeCap/mAh/g"],
            temp.loc[:idx_max, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_max + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_max + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 2)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.80,
    0.1,
    r"$\mathrm{1 \ M \ Zn(OTf){_2} \ in \ DMSO}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.98,
    0.1,
    r"$\mathrm{ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"h",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 I
subfig = fig.add_subfigure(gs[2, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, -0.24, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in zhs_138[2:3]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 30)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=6))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=3))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    1.0,
    0.1,
    r"$\mathrm{Zn(OTf){_2} \ in \ (DMSO + H_{2}O)}$, 1:3:8 wt.%",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.95,
    0.25,
    r"$\mathrm{ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"i",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 J
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.36, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in no_zhs_1M_02M[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 350)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.text(
    0.98,
    0.1,
    r"$\mathrm{1 \ M \ ZnSO{_4} + 0.2 \ M \ MnSO{_4}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"j",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 K
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.2, -0.36, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in no_zhs_1M_02M[1:2]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 3)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.75,
    0.1,
    r"$\mathrm{1 \ M \ Zn(OTf){_2} \ in \ DMSO}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.98,
    0.1,
    r"$\mathrm{No \ ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"k",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 L
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, -0.36, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in no_zhs_1M_02M[2:3]:
    for j, idx in enumerate(data.iloc[:, 0].unique()[1:3]):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[3],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[3],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 200)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=40))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=20))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

ax.text(
    0.7,
    0.1,
    r"$\mathrm{1 \ M \ ZnSO{_4} + 0.2 \ M \ MnSO{_4}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.98,
    0.1,
    r"$\mathrm{No \ ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"l",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 M
subfig = fig.add_subfigure(gs[4, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.48, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in no_zhs_138[0:1]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[7],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 350)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    0.98,
    0.10,
    r"$\mathrm{1 \ M \ ZnSO{_4} + 0.2 \ M \ MnSO{_4}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"m",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 N
subfig = fig.add_subfigure(gs[4, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.2, -0.48, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in no_zhs_138[1:2]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[0],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 2)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)

ax.text(
    0.75,
    0.1,
    r"$\mathrm{1 \ M \ Zn(OTf){_2} \ in \ DMSO}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.98,
    0.1,
    r"$\mathrm{No \ ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"n",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 O
subfig = fig.add_subfigure(gs[4, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, -0.48, 1.0, 1.0))
ax.set_box_aspect(0.8)

for data in no_zhs_138[2:3]:
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[: idx_min - 1, "SpeCap/mAh/g"],
            temp.loc[: idx_min - 1, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min + 1 :, "SpeCap/mAh/g"],
            temp.loc[idx_min + 1 :, "Voltage/V"],
            ls="-",
            lw=1.0,
            c=colors[1],  # type: ignore
            label=None,
            zorder=0,
        )

ax.set_xlim(0, 30)
# ax.set_xlabel(r'Capacity (mAh g$\mathrm{^{-1}_{MnO2}}$)', fontsize=9)  # noqa: ERA001
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=6))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=3))
ax.set_ylim(0.7, 1.9)
# ax.set_ylabel(r"Voltage (V)", fontsize=9)  # noqa: ERA001
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.1))
ax.tick_params(axis="both", which="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.text(
    1.0,
    0.1,
    r"$\mathrm{Zn(OTf){_2} \ in \ (DMSO + H_{2}O)}$, 1:3:8 wt.%",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    0.98,
    0.25,
    r"$\mathrm{No \ ZSH}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=9,
)
ax.text(
    -0.13,
    1.0,
    r"o",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

plt.figtext(
    -0.10,
    0.5,
    r"Voltage (V vs. Zn/Zn$^{2\!+}\!)$",
    fontsize=13,
    fontfamily="Arial",
    rotation="vertical",
    verticalalignment="center",
)
plt.figtext(
    0.55,
    -0.14,
    r"Capacity (mAh g$\mathrm{^{-1}_{MnO_2}}$)",
    fontsize=13,
    fontfamily="Arial",
    rotation="horizontal",
    horizontalalignment="center",
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_04_300.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_04_600.R1.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_04_600.R1.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_04_900.R1.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 03 TEM of pristine alpha MnO2 powder

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, data, color):
    asb = AnchoredSizeBar(
        ax.transData,
        size / data.axes_manager["y"].scale,
        "{} {}".format(size, data.axes_manager["y"].units),
        loc="lower left",
        pad=0.1,
        borderpad=0.5,
        sep=0.5,
        frameon=False,
        color=color,
        label_top=True,
    )
    ax.add_artist(asb)


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2023-EMCA\TEM\Data"  # noqa: E501, RUF001
)
path_eds = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2024-EMCA\EDS\Data\0002 - B8_HAADF_47000_x"  # noqa: E501, RUF001
)
path_tem2 = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\Powder\2024-EMCA\TEM\Result\0004 - B8_Ceta_640_kx Ceta"  # noqa: E501, RUF001
)
path_img = Path(
    DATA_ROOT / r"Zn-Mn\PaperUno\XRD\ExSitu\αMnO2\Pristine\Results\2022-ICMAB\PDF Card - 00-044-0141.tif"  # noqa: E501, RUF001
)
tem = hs.load(Path.joinpath(path_tem, r"0002 - 1214 Pristine MnO2 HAADF 58000 x.emd"))
tem2 = hs.load(Path.joinpath(path_tem2, r"0004 - B8_Ceta_640_kx Ceta.dm4"))
tem2_2 = hs.load(Path.joinpath(path_tem2, r"ifftSpaceB.dm4"))
tem2_3 = hs.load(Path.joinpath(path_tem2, r"SapceB.dm4"))
eds = hs.load(Path.joinpath(path_eds, r"0002 - B8_HAADF_47000_x.emd"))

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 5.5))
gs = gridspec.GridSpec(10, 12, width_ratios=np.ones(12), height_ratios=np.ones(10), wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:4, 0:4], zorder=0)
ax = subfig.add_axes((0, 0, 1, 1), zorder=0)
ax.set_aspect(1.0)
ax.imshow(tem.data, cmap="gray")
add_sizebar(ax, 500, tem, "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.03,
    1.0,
    r"a",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0:2, 4:8], zorder=0)
ax = subfig.add_axes((0.1, 0.01, 0.99, 0.99), zorder=0)
ax.set_aspect(1.0)
ax.imshow(eds[-2].data[:, 0:74], cmap="gray")
add_sizebar(ax, 100, eds[-2], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.03,
    1.0,
    r"b",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0:2, 8:12], zorder=0)
ax = subfig.add_axes((0.045, 0.01, 0.99, 0.99), zorder=0)
ax.set_aspect(1.0)
ax.imshow(eds[-6].data[:, 0:74], cmap=transparent_single_color_cmap(color0="k", color1=(27 / 255, 53 / 255, 209 / 255)))
add_sizebar(ax, 100, eds[-6], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2:4, 4:8], zorder=0)
ax = subfig.add_axes((0.1, 0, 0.99, 0.99), zorder=0)
ax.set_aspect(1.0)
ax.imshow(eds[-3].data[:, 0:74], cmap=transparent_single_color_cmap(color0="k", color1=(0, 237 / 255, 0)))
add_sizebar(ax, 100, eds[-3], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"O $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 E
subfig = fig.add_subfigure(gs[2:4, 8:12], zorder=0)
ax = subfig.add_axes((0.045, 0, 0.99, 0.99), zorder=0)
ax.set_aspect(1.0)
ax.imshow(eds[-4].data[:, 0:74], cmap=transparent_single_color_cmap(color0="k", color1=(236 / 255, 236 / 255, 0)))
add_sizebar(ax, 100, eds[-4], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 F
subfig = fig.add_subfigure(gs[4:10, 0:6], zorder=0)
ax = subfig.add_axes((-0.01, -0.05, 1, 1), zorder=0)
ax.set_aspect(1.0)

norm = Normalize(vmin=np.min(tem2.data), vmax=np.max(tem2.data))
ax.imshow(tem2.data, cmap="gray", norm=norm)
ax.set_axis_off()
ax.text(
    0.95,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.03,
    1.0,
    r"c",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 G
subfig = fig.add_subfigure(gs[4:7, 6:12], zorder=0)
ax = subfig.add_axes((0.1, 0, 0.9, 0.9), zorder=0)

ax.fill_between(
    x=tem2_3.real.axes_manager[-1].axis,
    y1=tem2_3.real.data,
    y2=0,
    ls="-",
    lw=1.0,
    cmap=transparent_single_color_cmap(color0="w", color1=(92 / 255, 191 / 255, 191 / 255)),
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9, labelpad=-0.5)
ax.set_xlim(0, 4.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(0, 250)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=100))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=50))
ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    top=False,
    left=False,
    right=False,
    labeltop=False,
    labelbottom=True,
    labelleft=False,
    labelright=False,
)
ax.text(
    -0.08,
    1.0,
    r"d",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 H
subfig = fig.add_subfigure(gs[7:10, 6:9], zorder=0)
ax = subfig.add_axes((0.2, -0.1, 0.85, 0.85), zorder=0)
ax.set_aspect(1.0)

norm = Normalize(vmin=np.min(tem2_2.real.data), vmax=np.max(tem2_2.real.data))
ax.imshow(tem2_2.real.data, cmap="gray", norm=norm)
add_sizebar(ax, 1, tem2_2.real, "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 I
subfig = fig.add_subfigure(gs[7:10, 9:12], zorder=0)
ax = subfig.add_axes((0.2, -0.1, 0.85, 0.85), zorder=0)
ax.set_aspect(1.0)
AA = plt.imread(
    path_img,
    format="tif",
)
ax.imshow(AA, origin="upper")
ax.set_axis_off()
ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    top=False,
    left=False,
    right=False,
    labeltop=False,
    labelbottom=True,
    labelleft=False,
    labelright=False,
)

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_03_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_03_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_03_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_03_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure_SI 02 XRD of pristine αMnO2 powder

In [ ]:
# 读取数据
path_xrd = Path(DATA_ROOT / r"Zn-Mn\PaperUno\XRD\ExSitu\αMnO2\Pristine\Results\2022-ICMAB")
xrd = pd.read_csv(
    Path.joinpath(path_xrd, r"αMnO2_Pristine_2022.UXD"), sep=r"\s+", index_col=None, header=0, comment="#"
)
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDF-00-044-0141.csv"), sep=r",", index_col=None, header=0, comment="#")
path_img = Path.joinpath(path_xrd, r"PDF Card - 00-044-0141.tif")

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d = ax.plot(xrd.iloc[:, 0], xrd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\alpha$-MnO${_2}$")  # type: ignore
stem = ax.stem(
    pdf.iloc[:, 0],
    pdf.iloc[:, 2] * 0.5,
    linefmt=colors[3],  # type: ignore
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-044-0141",
)

ax.set_xlim(5, 80)
ax.set_xlabel(r"2-Theta$\mathrm{\ _{Cu}}$ (degree)", fontsize=9)
ax.set_ylim(0, 250)
ax.set_ylabel("Relative Intensity (arb.u.)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5))
# ax.set(yticks=[])  # noqa: ERA001
ax.tick_params(axis="both", labelsize=9, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d,
    loc="upper right",
    bbox_to_anchor=(1.02, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
)
ax.text(
    0.98,
    0.85,
    r"PDF#00-044-0141",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)

axins = ax.inset_axes((0.28, 0.45, 0.45, 0.45))
AA = plt.imread(path_img, format="tif")
axins.imshow(AA, origin="upper", zorder=2)
axins.set_axis_off()

plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_02_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_02_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_02_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperUno_FigureSI_02_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()